In [15]:
import pandas as pd
import os
from zipfile import ZipFile
import requests

# ----------------------------
# 1. Download & Extract MovieLens 1M
# ----------------------------
dataset_url = "https://files.grouplens.org/datasets/movielens/ml-1m.zip"
dataset_zip = "ml-1m.zip"
extracted_path = "ml-1m"

if not os.path.exists(extracted_path):
    print("Downloading MovieLens 1M dataset...")
    r = requests.get(dataset_url)
    with open(dataset_zip, "wb") as f:
        f.write(r.content)
    with ZipFile(dataset_zip, 'r') as zip_ref:
        zip_ref.extractall(".")
    print("Download and extraction done.")

# ----------------------------
# 2. Load all files
# ----------------------------
ratings_file = os.path.join(extracted_path, "ratings.dat")
movies_file = os.path.join(extracted_path, "movies.dat")
users_file  = os.path.join(extracted_path, "users.dat")

ratings = pd.read_csv(
    ratings_file, sep="::", engine="python",
    names=["UserID","MovieID","Rating","Timestamp"]
)
movies = pd.read_csv(
    movies_file, sep="::", engine="python",
    names=["MovieID","Title","Genres"], encoding="latin-1"
)
users = pd.read_csv(
    users_file, sep="::", engine="python",
    names=["UserID","Gender","Age","Occupation","Zip-code"]
)

# ----------------------------
# 3. Convert timestamp
# ----------------------------
ratings['Timestamp'] = pd.to_datetime(ratings['Timestamp'], unit='s')

# ----------------------------
# 4. Check for duplicates
# ----------------------------
print("Duplicate rows:")
print(f"Ratings: {ratings.duplicated().sum()}")
print(f"Movies:  {movies.duplicated().sum()}")
print(f"Users:   {users.duplicated().sum()}")

# Drop duplicates
ratings = ratings.drop_duplicates().reset_index(drop=True)
movies  = movies.drop_duplicates().reset_index(drop=True)
users   = users.drop_duplicates().reset_index(drop=True)

# ----------------------------
# 5. Check for missing values in detail
# ----------------------------
def report_missing(df, name):
    total_missing = df.isnull().sum().sum()
    if total_missing == 0:
        print(f"No missing values in {name}.")
    else:
        print(f"Missing values in {name}:")
        for col in df.columns:
            missing_count = df[col].isnull().sum()
            if missing_count > 0:
                print(f" - Column '{col}': {missing_count} missing")
        print("Rows with missing values:")
        print(df[df.isnull().any(axis=1)])

report_missing(ratings, "ratings")
report_missing(movies, "movies")
report_missing(users, "users")

# ----------------------------
# 6. Filter ratings to May → Dec 2000
# ----------------------------
start_date = '2000-05-01'
end_date   = '2000-12-31'
ratings = ratings[(ratings['Timestamp'] >= start_date) & (ratings['Timestamp'] <= end_date)].reset_index(drop=True)
print("\nRatings after filtering May → Dec 2000:", ratings.shape)


Duplicate rows:
Ratings: 0
Movies:  0
Users:   0
No missing values in ratings.
No missing values in movies.
No missing values in users.

Ratings after filtering May → Dec 2000: (891189, 4)


In [16]:
# ----------------------------
# Create monthly slices
# ----------------------------
monthly_bins = pd.date_range(start=start_date, end='2001-01-01', freq='MS')
ratings['SliceNum'] = pd.cut(ratings['Timestamp'], bins=monthly_bins, right=False).cat.codes + 1

# ----------------------------
# Inspect each slice
# ----------------------------
slice_summary = []

for s in sorted(ratings['SliceNum'].unique()):
    slice_df = ratings[ratings['SliceNum'] == s]
    num_ratings = slice_df.shape[0]
    num_movies  = slice_df['MovieID'].nunique()
    num_users   = slice_df['UserID'].nunique()
    slice_summary.append({
        'SliceNum': s,
        'Ratings': num_ratings,
        'UniqueMovies': num_movies,
        'UniqueUsers': num_users
    })

slice_summary_df = pd.DataFrame(slice_summary)
print(slice_summary_df)


   SliceNum  Ratings  UniqueMovies  UniqueUsers
0         1    67437          2920          486
1         2    54486          2913          508
2         3    90334          3135          778
3         4   182109          3298         1310
4         5    52421          3083          576
5         6    42294          2993          500
6         7   290793          3552         2357
7         8   111315          3331         1215


In [17]:
import os
import pandas as pd

# ----------------------------
# 1. Create folder for slice metadata
# ----------------------------
os.makedirs("slice_metadata", exist_ok=True)

# ----------------------------
# 2. Split movies into popularity tiers per slice
# ----------------------------
high_pct, mid_pct, low_pct = 0.2, 0.3, 0.5  # 20/30/50 split

for s in sorted(ratings['SliceNum'].unique()):
    slice_df = ratings[ratings['SliceNum'] == s]
    movie_counts = slice_df.groupby('MovieID')['Rating'].count().sort_values(ascending=False)
    total_movies = len(movie_counts)
    
    # Determine cutoff indices
    high_cut  = int(total_movies * high_pct)
    mid_cut   = high_cut + int(total_movies * mid_pct)
    
    # Assign tiers
    movie_tiers = pd.DataFrame({'MovieID': movie_counts.index})
    movie_tiers['PopTier'] = 'low'  # default
    movie_tiers.loc[:high_cut-1, 'PopTier'] = 'high'
    movie_tiers.loc[high_cut:mid_cut-1, 'PopTier'] = 'medium'
    
    # Save slice metadata
    movie_tiers.to_csv(f"slice_metadata/movie_pop_tiers_slice_{s}.csv", index=False)
    print(f"Slice {s}: {total_movies} movies → High:{high_cut}, Medium:{mid_cut-high_cut}, Low:{total_movies-mid_cut}")


Slice 1: 2920 movies → High:584, Medium:876, Low:1460
Slice 2: 2913 movies → High:582, Medium:873, Low:1458
Slice 3: 3135 movies → High:627, Medium:940, Low:1568
Slice 4: 3298 movies → High:659, Medium:989, Low:1650
Slice 5: 3083 movies → High:616, Medium:924, Low:1543
Slice 6: 2993 movies → High:598, Medium:897, Low:1498
Slice 7: 3552 movies → High:710, Medium:1065, Low:1777
Slice 8: 3331 movies → High:666, Medium:999, Low:1666


In [18]:
# ----------------------------
# User type assignment per slice
# ----------------------------
import os
import pandas as pd

slice_nums = sorted(ratings['SliceNum'].unique())
os.makedirs("slice_metadata", exist_ok=True)

threshold = 0.7  # HighProportion threshold to classify mainstream

for s in slice_nums:
    slice_df = ratings[ratings['SliceNum'] == s].copy()
    if slice_df.empty: 
        continue

    # Load movie popularity tiers for this slice
    movie_tiers = pd.read_csv(f"slice_metadata/movie_pop_tiers_slice_{s}.csv")
    movie_tier_map = dict(zip(movie_tiers.MovieID, movie_tiers.PopTier))

    # Map each rating to movie tier
    slice_df['Tier'] = slice_df['MovieID'].map(movie_tier_map)

    # Count number of ratings per tier per user
    tier_counts = slice_df.groupby(['UserID', 'Tier']).size().unstack(fill_value=0)
    # Ensure all columns exist
    for col in ['high', 'medium', 'low']:
        if col not in tier_counts.columns:
            tier_counts[col] = 0

    # Compute HighProportion
    tier_counts['Total'] = tier_counts['high'] + tier_counts['medium'] + tier_counts['low']
    tier_counts['HighProportion'] = tier_counts['high'] / tier_counts['Total']

    # Assign user type
    tier_counts['UserType'] = tier_counts['HighProportion'].apply(
        lambda x: 'mainstream' if x >= threshold else 'niche'
    )

    # Save result
    user_type_df = tier_counts[['UserType']].reset_index()
    user_type_df.to_csv(f"slice_metadata/user_labels_slice_{s}.csv", index=False)
    
    print(f"Slice {s}: Total Users = {len(user_type_df)}, Mainstream = {sum(user_type_df.UserType=='mainstream')}, Niche = {sum(user_type_df.UserType=='niche')}")


Slice 1: Total Users = 486, Mainstream = 233, Niche = 253
Slice 2: Total Users = 508, Mainstream = 239, Niche = 269
Slice 3: Total Users = 778, Mainstream = 332, Niche = 446
Slice 4: Total Users = 1310, Mainstream = 598, Niche = 712
Slice 5: Total Users = 576, Mainstream = 276, Niche = 300
Slice 6: Total Users = 500, Mainstream = 223, Niche = 277
Slice 7: Total Users = 2357, Mainstream = 1420, Niche = 937
Slice 8: Total Users = 1215, Mainstream = 596, Niche = 619


In [19]:
import pandas as pd
import os
import torch

# ----------------------------
# 1. KG construction per slice with relation types and mappings
# ----------------------------
slice_nums = sorted(ratings['SliceNum'].unique())
os.makedirs("slice_metadata/kg", exist_ok=True)

# Define relation types
REL_USER_MOVIE = 0
REL_MOVIE_GENRE = 1
REL_MOVIE_POPTIER = 2
REL_USER_USERTYPE = 3

for s in slice_nums:
    slice_df = ratings[ratings['SliceNum'] == s].copy()
    if slice_df.empty:
        continue
    
    print(f"\nConstructing KG for slice {s}...")

    # Load slice-specific metadata
    movie_tiers = pd.read_csv(f"slice_metadata/movie_pop_tiers_slice_{s}.csv")
    user_types  = pd.read_csv(f"slice_metadata/user_labels_slice_{s}.csv")
    
    # Extract unique nodes
    users = slice_df['UserID'].unique()
    movies_slice = slice_df['MovieID'].unique()
    genres = set()
    for g in movies['Genres']:
        genres.update(g.split('|'))
    genres = list(genres)
    
    pop_tiers = ['high', 'medium', 'low']
    utypes = ['mainstream', 'niche']
    
    # ----------------------------
    # Slice-local IDs
    user2nid = {u: i for i, u in enumerate(users)}
    movie2nid = {m: i + len(users) for i, m in enumerate(movies_slice)}
    genre2nid = {g: i + len(users) + len(movies_slice) for i, g in enumerate(genres)}
    pop2nid = {p: i + len(users) + len(movies_slice) + len(genres) for i, p in enumerate(pop_tiers)}
    utype2nid = {ut: i + len(users) + len(movies_slice) + len(genres) + len(pop_tiers) for i, ut in enumerate(utypes)}
    
    # Save mapping tables for hybrid warm-start later
    pd.DataFrame(list(user2nid.items()), columns=['UserID','NodeID']).to_csv(f"slice_metadata/kg/user_map_slice_{s}.csv", index=False)
    pd.DataFrame(list(movie2nid.items()), columns=['MovieID','NodeID']).to_csv(f"slice_metadata/kg/movie_map_slice_{s}.csv", index=False)
    
    # ----------------------------
    # Build edges with relation types
    edges = []

    # User ↔ Movie
    for row in slice_df.itertuples():
        uid, mid = user2nid[row.UserID], movie2nid[row.MovieID]
        edges.append((uid, mid, REL_USER_MOVIE))
        edges.append((mid, uid, REL_USER_MOVIE))  # bi-directional

    # Movie ↔ Genre
    for row in movies.itertuples():
        if row.MovieID not in movie2nid:
            continue
        mid = movie2nid[row.MovieID]
        for g in row.Genres.split('|'):
            edges.append((mid, genre2nid[g], REL_MOVIE_GENRE))
            edges.append((genre2nid[g], mid, REL_MOVIE_GENRE))

    # Movie ↔ PopTier
    for row in movie_tiers.itertuples():
        mid = movie2nid[row.MovieID]
        edges.append((mid, pop2nid[row.PopTier], REL_MOVIE_POPTIER))
        edges.append((pop2nid[row.PopTier], mid, REL_MOVIE_POPTIER))

    # User ↔ UserType
    for row in user_types.itertuples():
        uid = user2nid[row.UserID]
        edges.append((uid, utype2nid[row.UserType], REL_USER_USERTYPE))
        edges.append((utype2nid[row.UserType], uid, REL_USER_USERTYPE))
    
    # Save edges with relation types
    edges_df = pd.DataFrame(edges, columns=['Src','Dst','RelType'])
    edges_df.to_csv(f"slice_metadata/kg/kg_edges_slice_{s}.csv", index=False)
    
    print(f"Slice {s} KG: {len(edges_df)} edges, {len(user2nid)} users, {len(movie2nid)} movies, {len(genre2nid)} genres, {len(pop2nid)} pop tiers, {len(utype2nid)} user types")



Constructing KG for slice 1...
Slice 1 KG: 151890 edges, 486 users, 2920 movies, 18 genres, 3 pop tiers, 2 user types

Constructing KG for slice 2...
Slice 2 KG: 126078 edges, 508 users, 2913 movies, 18 genres, 3 pop tiers, 2 user types

Constructing KG for slice 3...
Slice 3 KG: 199352 edges, 778 users, 3135 movies, 18 genres, 3 pop tiers, 2 user types

Constructing KG for slice 4...
Slice 4 KG: 384750 edges, 1310 users, 3298 movies, 18 genres, 3 pop tiers, 2 user types

Constructing KG for slice 5...
Slice 5 KG: 122904 edges, 576 users, 3083 movies, 18 genres, 3 pop tiers, 2 user types

Constructing KG for slice 6...
Slice 6 KG: 102034 edges, 500 users, 2993 movies, 18 genres, 3 pop tiers, 2 user types

Constructing KG for slice 7...
Slice 7 KG: 605404 edges, 2357 users, 3552 movies, 18 genres, 3 pop tiers, 2 user types

Constructing KG for slice 8...
Slice 8 KG: 243150 edges, 1215 users, 3331 movies, 18 genres, 3 pop tiers, 2 user types


PROJECT README
Temporal Knowledge Graphs for Recommender Systems
================================================

1. What this project is about
-----------------------------

This project studies how user preferences and item popularity change over time,
and how recommender models can adapt to those changes instead of treating data as static.

Most recommender systems:
- Merge all interactions into one dataset
- Assume user behavior and item popularity are fixed
- Ignore temporal structure

That is unrealistic.

In this project, we:
- Split user–item interactions into time slices
- Build a separate Knowledge Graph (KG) for each slice
- Train models slice by slice
- Compare a baseline recommender with a more expressive final model

The goal is to evaluate whether modeling structure and time actually improves recommendations.


2. Why use a Knowledge Graph
----------------------------

A standard user–item matrix only captures:
“User A interacted with Movie B”.

A Knowledge Graph allows us to represent richer structure:
- Users have behavioral types (mainstream vs niche)
- Movies have genres
- Movies have popularity levels
- All of these can change over time

By encoding interactions as a graph:
- Simple graph models can be used as baselines
- Relation-aware models can exploit edge types
- The same data supports multiple model families


3. Why time slicing
-------------------

User behavior in May 2000 is not the same as in December 2000.

Instead of building one global graph, we:
- Split ratings into monthly time slices
- Build one KG per slice
- Allow users and movies to appear, disappear, or change context

This enables:
- Temporal evaluation
- Embedding warm-starting
- Studying preference drift


4. Models trained later
-----------------------

This preprocessing supports two categories of models:

1. Baseline model
   - Treats all edges as the same
   - Uses only graph structure
   - Ignores relation types

2. Final model
   - Uses relation types
   - Distinguishes different kinds of edges
   - Exploits richer KG structure

Both models use the same preprocessed data.


5. Dataset
----------

- MovieLens 1M
- Ratings filtered to May 2000 – December 2000
- Monthly time slices


6. Outputs per time slice
------------------------

For each slice, the following are created:
- Movie popularity tiers
- User behavioral types
- Slice-specific Knowledge Graph
- Node ID mappings for temporal alignment

All outputs are stored as CSV files.


7. Directory structure
----------------------

.
├── slice_metadata/
│   ├── movie_pop_tiers_slice_1.csv
│   ├── user_labels_slice_1.csv
│   ├── ...
│   └── kg/
│       ├── kg_edges_slice_1.csv
│       ├── user_map_slice_1.csv
│       ├── movie_map_slice_1.csv
│       └── ...


8. Movie popularity tiers
-------------------------

Files:
slice_metadata/movie_pop_tiers_slice_{s}.csv

Description:
- Movies are ranked by number of ratings within a slice
- Assigned to popularity tiers:
  - high
  - medium
  - low

These tiers capture short-term popularity, not global popularity.


9. User behavioral types
------------------------

Files:
slice_metadata/user_labels_slice_{s}.csv

Description:
- Users are labeled based on what they consume in that slice
- Users who mostly rate popular movies → mainstream
- Others → niche

User types can change across slices.


10. Knowledge Graphs
--------------------

Each slice has its own Knowledge Graph.


10.1 Edge list
--------------

Files:
slice_metadata/kg/kg_edges_slice_{s}.csv

Columns:
- Src     : source node ID
- Dst     : destination node ID
- RelType : relation type ID

All edges are stored as bidirectional.


10.2 Relation types
-------------------

Relation ID meanings:

0 → User ↔ Movie  
1 → Movie ↔ Genre  
2 → Movie ↔ Popularity Tier  
3 → User ↔ User Type  

Baseline models ignore RelType.
Final models use RelType.


11. Node IDs
------------

Node IDs are:
- Integers
- Unique within a slice
- Assigned in non-overlapping ranges by node type

This guarantees that users, movies, genres, tiers, and user types
cannot be confused with each other.


12. Node ID mappings (important)
--------------------------------

Files:
- slice_metadata/kg/user_map_slice_{s}.csv
- slice_metadata/kg/movie_map_slice_{s}.csv

These map original UserID / MovieID to slice-local node IDs.

They are required for:
- Matching entities across slices
- Warm-starting embeddings
- Temporal stabilization during training


13. What is NOT stored
----------------------

- No trained models
- No tensors
- No pickled Python objects
- No framework-specific formats

Everything is reconstructed from CSVs in training notebooks.


14. How a new training session should start
-------------------------------------------

A fresh session should:
1. Select a slice
2. Load KG edges and mappings
3. Convert data to tensors as needed
4. Train baseline and final models
5. Optionally align embeddings across slices


15. Summary
-----------

This project builds time-aware Knowledge Graphs from MovieLens data
to compare a baseline recommender against a relation-aware model
while explicitly modeling temporal dynamics.


K = 10

In [20]:
import os
# CRITICAL FIX: Deterministic behavior for reproducibility
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import scipy.sparse as sp

# ----------------------------
# 1. Device & Setup
# ----------------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
np.random.seed(42)
torch.use_deterministic_algorithms(True)

# ----------------------------
# 2. Metric Helpers (Unchanged)
# ----------------------------
def get_gini(recommendation_counts):
    arr = np.sort(np.array(recommendation_counts))
    if np.sum(arr) == 0: return 0.0
    n = len(arr)
    return (n + 1 - 2 * np.sum(np.cumsum(arr)) / np.sum(arr)) / n

def compute_metrics(model, adj_dict, test_df, train_df, user_map, item_map, 
                    user_labels, movie_tiers, K=10):
    model.eval()
    with torch.no_grad():
        # KGCN Forward Pass
        final_emb = model(adj_dict)
        
        u_nodes = list(user_map.values())
        i_nodes = list(item_map.values())
        
        valid_test = test_df[test_df['UserID'].isin(user_map.keys()) & 
                             test_df['MovieID'].isin(item_map.keys())].copy()
        if valid_test.empty: return None, 0

        valid_test['u_node'] = valid_test['UserID'].map(user_map)
        valid_test['i_node'] = valid_test['MovieID'].map(item_map)
        
        user_gt = valid_test.groupby('u_node')['i_node'].apply(set).to_dict()
        test_u_nodes = list(user_gt.keys())
        
        test_u_tensor = torch.LongTensor(test_u_nodes).to(device)
        all_i_tensor = torch.LongTensor(i_nodes).to(device)
        
        user_embs = final_emb[test_u_tensor]
        item_embs = final_emb[all_i_tensor]
        
        scores = torch.matmul(user_embs, item_embs.t())
        
        train_interactions = train_df.groupby('UserID')['MovieID'].apply(list).to_dict()
        local_idx_to_node_id = {idx: node_id for idx, node_id in enumerate(i_nodes)}
        node_to_u_id_map = {v: k for k, v in user_map.items()}
        
        for idx, u_node in enumerate(test_u_nodes):
            u_id = node_to_u_id_map[u_node]
            if u_id in train_interactions:
                raw_seen = train_interactions[u_id]
                seen_nodes = set(item_map[m] for m in raw_seen if m in item_map)
                mask_indices = [k for k, v in local_idx_to_node_id.items() if v in seen_nodes]
                if mask_indices:
                    scores[idx, mask_indices] = -1e9

        _, top_indices = torch.topk(scores, K, dim=1)
        top_indices = top_indices.cpu().numpy()
        
        recall_arr, ndcg_arr = [], []
        group_recalls = {'mainstream': [], 'niche': []}
        pop_counts = {'mainstream': {'high': 0, 'medium': 0, 'low': 0}, 
                      'niche': {'high': 0, 'medium': 0, 'low': 0}}
        total_recs = {'mainstream': 0, 'niche': 0}
        
        node_to_m_id = {v: k for k, v in item_map.items()}
        all_item_counts = np.zeros(len(i_nodes))

        for idx, u_node in enumerate(test_u_nodes):
            truth = user_gt[u_node]
            rec_node_ids = [local_idx_to_node_id[x] for x in top_indices[idx]]
            
            hits = len(set(rec_node_ids) & truth)
            recall_arr.append(hits / len(truth))
            
            dcg, idcg = 0.0, 0.0
            for k, r in enumerate(rec_node_ids):
                if r in truth: dcg += 1.0 / np.log2(k + 2)
            for k in range(min(len(truth), K)): idcg += 1.0 / np.log2(k + 2)
            ndcg_arr.append(dcg/idcg if idcg > 0 else 0)
            
            u_id = node_to_u_id_map[u_node]
            u_type = user_labels.get(u_id, 'niche')
            group_recalls[u_type].append(hits / len(truth))
            total_recs[u_type] += K
            
            for local_idx, r_node in zip(top_indices[idx], rec_node_ids):
                all_item_counts[local_idx] += 1
                raw_mid = node_to_m_id.get(r_node)
                tier = movie_tiers.get(raw_mid, 'low')
                pop_counts[u_type][tier] += 1
        
        raw_exp = {grp: {t: (pop_counts[grp][t]/total_recs[grp]) if total_recs[grp]>0 else 0.0 for t in ['low','medium','high']} for grp in ['mainstream','niche']}
        formatted_pop = {grp: {t: f"{(raw_exp[grp][t]*100):.2f}%" for t in ['low','medium','high']} for grp in ['mainstream','niche']}
        
        eo = abs(np.mean(group_recalls['mainstream']) - np.mean(group_recalls['niche'])) if group_recalls['mainstream'] else 0
        dp = (abs(raw_exp['mainstream']['low'] - raw_exp['niche']['low']) +
              abs(raw_exp['mainstream']['medium'] - raw_exp['niche']['medium']) +
              abs(raw_exp['mainstream']['high'] - raw_exp['niche']['high'])) / 3.0
        gini_val = get_gini(all_item_counts)
        
        return {
            'Recall': np.mean(recall_arr), 'NDCG': np.mean(ndcg_arr), 'EO': eo, 'DP': dp, 
            'Gini': gini_val, 'PopExposure': formatted_pop, 'RawExposure': raw_exp
        }, len(test_u_nodes)

# ----------------------------
# 3. Model Logic: True KGCN (R-GCN Style)
# ----------------------------
class RGCNLayer(nn.Module):
    """
    True Relation-Aware Layer.
    Uses a specific Weight Matrix (W_r) for each relation type.
    Equation: h_u = Tanh( W_0 * h_u + Sum_r( Sum_v( W_r * h_v ) ) )
    This is 'fat' because it has params: num_relations * dim * dim.
    """
    def __init__(self, in_dim, out_dim, num_rels):
        super().__init__()
        self.in_dim = in_dim
        self.out_dim = out_dim
        self.num_rels = num_rels
        
        # The 'Fat' Part: A unique weight matrix for every relation type
        # We use nn.ParameterTensor for efficient lookup
        self.weight = nn.Parameter(torch.Tensor(num_rels, in_dim, out_dim))
        
        # Self-loop weight (how much the node keeps its own identity)
        self.w_self = nn.Linear(in_dim, out_dim)
        
        # Initialization
        nn.init.xavier_uniform_(self.weight)

    def forward(self, x, adj_dict):
        # x: [num_nodes, in_dim]
        
        # 1. Start with Self-Loop info
        out = self.w_self(x)
        
        # 2. Iterate over relations (Relation-Aware Aggregation)
        #  
        for rel_type, adj in adj_dict.items():
            # Safety check for relation index
            if rel_type >= self.num_rels: continue 
            
            # A. Retrieve the specific Weight Matrix for this relation
            w_r = self.weight[rel_type] # [in_dim, out_dim]
            
            # B. Transform node features specifically for this relation
            #    (Dense Matrix Mul -> GPU Efficient)
            x_transformed = torch.matmul(x, w_r) # [num_nodes, out_dim]
            
            # C. Propagate using the sparse adjacency
            #    (Sparse Matrix Mul)
            #    Aggregates the transformed neighbors
            out_r = torch.sparse.mm(adj, x_transformed)
            
            # Add to accumulator
            out = out + out_r
            
        return torch.tanh(out)

class TrueKGCN(nn.Module):
    def __init__(self, num_nodes, num_relations, embedding_dim=64, n_layers=2):
        super().__init__()
        self.node_emb = nn.Embedding(num_nodes, embedding_dim)
        
        # Stack of RGCN Layers
        self.layers = nn.ModuleList([
            RGCNLayer(embedding_dim, embedding_dim, num_relations)
            for _ in range(n_layers)
        ])
        
        nn.init.xavier_uniform_(self.node_emb.weight)

    def forward(self, adj_dict):
        x = self.node_emb.weight
        for layer in self.layers:
            x = layer(x, adj_dict)
        return x

def build_relational_adjacency(kg_edges_df, num_nodes):
    # Same logic, just builds the sparse matrices
    adj_dict = {}
    rel_types = kg_edges_df['RelType'].unique()
    
    for r in rel_types:
        sub_df = kg_edges_df[kg_edges_df['RelType'] == r]
        src = sub_df['Src'].values
        dst = sub_df['Dst'].values
        indices = np.vstack((src, dst))
        values = np.ones(len(src))
        
        i = torch.LongTensor(indices)
        v = torch.FloatTensor(values)
        shape = (num_nodes, num_nodes)
        
        adj = torch.sparse_coo_tensor(i, v, shape).coalesce()
        
        # Row-Normalize
        deg = torch.sparse.sum(adj, dim=1).to_dense()
        deg_inv = deg.pow(-1)
        deg_inv[deg_inv == float('inf')] = 0
        
        adj = torch.sparse_coo_tensor(i, deg_inv[i[0]], shape).to(device)
        adj_dict[r] = adj
        
    return adj_dict

# ----------------------------
# 4. Feedback Training Loop (Unchanged Workflow)
# ----------------------------
LR, EPOCHS, BATCH_SIZE = 0.001, 50, 2048
MAX_RETRIES = 10 

MAIN_TARGET_HIGH = (0.45, 0.55)
MAIN_TARGET_LOW  = (0.15, 0.25)
NICHE_TARGET_HIGH = (0.35, 0.45)
NICHE_TARGET_LOW  = (0.35, 0.45)

w_main_vals = [1.5, 1.0, 0.1]
w_niche_vals = [2.0, 1.0, 0.3]

prev_model_state = None
prev_node_map = {} 

print(f"{'Slice Pair':<13} | {'Users':<6} | {'Recall':<8} | {'NDCG':<8} | {'EO':<8} | {'DP':<8} | {'Gini':<8}")
print("-" * 105)

slice_nums = sorted(ratings['SliceNum'].unique())

for s in slice_nums[:-1]: 
    train_df = ratings[ratings['SliceNum'] == s].copy()
    test_df  = ratings[ratings['SliceNum'] == s+1].copy()
    
    user_map_df = pd.read_csv(f"slice_metadata/kg/user_map_slice_{s}.csv")
    movie_map_df = pd.read_csv(f"slice_metadata/kg/movie_map_slice_{s}.csv")
    kg_edges_df = pd.read_csv(f"slice_metadata/kg/kg_edges_slice_{s}.csv")
    
    u_map = dict(zip(user_map_df.UserID, user_map_df.NodeID))
    i_map = dict(zip(movie_map_df.MovieID, movie_map_df.NodeID))
    
    current_node_lookup = {}
    for k, v in u_map.items(): current_node_lookup[f"u_{k}"] = v
    for k, v in i_map.items(): current_node_lookup[f"m_{k}"] = v

    user_labels = pd.read_csv(f"slice_metadata/user_labels_slice_{s}.csv").set_index('UserID')['UserType'].to_dict()
    movie_tiers = pd.read_csv(f"slice_metadata/movie_pop_tiers_slice_{s}.csv").set_index('MovieID')['PopTier'].to_dict()

    max_node_id = max(kg_edges_df['Src'].max(), kg_edges_df['Dst'].max()) + 1
    
    # CRITICAL: Determine max relation ID for weight matrix dimensioning
    max_rel_id = kg_edges_df['RelType'].max() + 1
    
    adj_dict = build_relational_adjacency(kg_edges_df, max_node_id)
    
    train_df = train_df[train_df['UserID'].isin(u_map) & train_df['MovieID'].isin(i_map)]
    train_user_tensor = torch.LongTensor(train_df['UserID'].map(u_map).values).to(device)
    train_item_tensor = torch.LongTensor(train_df['MovieID'].map(i_map).values).to(device)
    num_train_samples = len(train_user_tensor)

    tier_idx_map = {'low': 0, 'medium': 1, 'high': 2}
    u_type_idx_map = {'mainstream': 0, 'niche': 1}
    
    item_tier_tensor = torch.zeros(max_node_id, dtype=torch.long, device=device)
    for mid, node_id in i_map.items():
        tier = movie_tiers.get(mid, 'low')
        item_tier_tensor[node_id] = tier_idx_map[tier]

    user_type_tensor = torch.zeros(max_node_id, dtype=torch.long, device=device)
    for uid, node_id in u_map.items():
        ut = user_labels.get(uid, 'niche')
        user_type_tensor[node_id] = u_type_idx_map[ut]

    retry_count = 1
    success = False
    
    while retry_count <= MAX_RETRIES and not success:
        # INSTANTIATE TRUE KGCN
        model = TrueKGCN(max_node_id, max_rel_id).to(device)
        
        if prev_model_state:
            with torch.no_grad():
                # Load Layer Weights (The 'Fat' parts)
                # We iterate layers to load W_r safely
                for i, layer in enumerate(model.layers):
                    w_name = f'layers.{i}.weight'
                    ws_name = f'layers.{i}.w_self.weight'
                    bs_name = f'layers.{i}.w_self.bias'
                    
                    if w_name in prev_model_state:
                        # Slice to fit in case num_rels changed (unlikely but safe)
                        prev_w = prev_model_state[w_name]
                        curr_rels = layer.weight.shape[0]
                        prev_rels = prev_w.shape[0]
                        sz = min(curr_rels, prev_rels)
                        layer.weight.data[:sz] = prev_w[:sz].to(device)
                        
                    if ws_name in prev_model_state:
                        layer.w_self.weight.data = prev_model_state[ws_name].to(device)
                        layer.w_self.bias.data = prev_model_state[bs_name].to(device)

                # Load node embeddings
                prev_nodes = prev_model_state['node_emb.weight'].to(device)
                for entity_key, curr_id in current_node_lookup.items():
                    if entity_key in prev_node_map:
                        prev_id = prev_node_map[entity_key]
                        if prev_id < prev_nodes.shape[0]:
                            model.node_emb.weight.data[curr_id] = prev_nodes[prev_id]

        optimizer = optim.Adam(model.parameters(), lr=LR)
        weight_matrix = torch.tensor([w_main_vals, w_niche_vals], dtype=torch.float, device=device)
        
        model.train()
        
        # Optimization: Move negative sampling pool to GPU
        all_item_nodes = torch.LongTensor(list(i_map.values())).to(device)
        
        for epoch in range(EPOCHS):
            rand_indices = torch.randint(0, len(all_item_nodes), (num_train_samples,), device=device)
            neg_nodes = all_item_nodes[rand_indices]
            
            idxs = torch.randperm(num_train_samples, device=device)
            
            for i in range(0, num_train_samples, BATCH_SIZE):
                b = idxs[i:i+BATCH_SIZE]
                
                u_b = train_user_tensor[b]
                p_b = train_item_tensor[b]
                n_b = neg_nodes[b]
                
                final_emb = model(adj_dict)
                
                pos_scores = (final_emb[u_b] * final_emb[p_b]).sum(1)
                neg_scores = (final_emb[u_b] * final_emb[n_b]).sum(1)
                bpr = -torch.nn.functional.logsigmoid(pos_scores - neg_scores)
                
                w_p = weight_matrix[user_type_tensor[u_b], item_tier_tensor[p_b]]
                loss = (bpr * w_p).mean()
                
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
        
        metrics, count = compute_metrics(
            model, adj_dict, test_df, train_df, 
            u_map, i_map, user_labels, movie_tiers
        )
        
        if metrics:
            m_h = metrics['RawExposure']['mainstream']['high']
            m_l = metrics['RawExposure']['mainstream']['low']
            n_h = metrics['RawExposure']['niche']['high']
            n_l = metrics['RawExposure']['niche']['low']
            
            needs_retry = False
            if m_h < MAIN_TARGET_HIGH[0]: w_main_vals[2] += 0.05; needs_retry = True
            elif m_h > MAIN_TARGET_HIGH[1]: w_main_vals[2] = max(0, w_main_vals[2] - 0.05); needs_retry = True
            if m_l < MAIN_TARGET_LOW[0]: w_main_vals[0] += 0.2; needs_retry = True
            elif m_l > MAIN_TARGET_LOW[1]: w_main_vals[0] = max(0, w_main_vals[0] - 0.2); needs_retry = True
            if n_h < NICHE_TARGET_HIGH[0]: w_niche_vals[2] += 0.05; needs_retry = True
            elif n_h > NICHE_TARGET_HIGH[1]: w_niche_vals[2] = max(0, w_niche_vals[2] - 0.05); needs_retry = True
            if n_l < NICHE_TARGET_LOW[0]: w_niche_vals[0] += 0.2; needs_retry = True
            elif n_l > NICHE_TARGET_LOW[1]: w_niche_vals[0] = max(0, w_niche_vals[0] - 0.2); needs_retry = True

            if not needs_retry or retry_count == MAX_RETRIES:
                print(f"{s} -> {s+1:<8} | {count:<6} | {metrics['Recall']:.4f}    | {metrics['NDCG']:.4f}    | {metrics['EO']:.4f}    | {metrics['DP']:.4f}    | {metrics['Gini']:.4f}")
                w_m_str = "/".join([f"{v:.1f}" for v in w_main_vals])
                w_n_str = "/".join([f"{v:.1f}" for v in w_niche_vals])
                print(f"PopExposure: {metrics['PopExposure']} W: {{M:{w_m_str}, N:{w_n_str}}}")
                
                success = True
                prev_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                prev_node_map = current_node_lookup.copy()
            else:
                retry_count += 1
        else:
            break

Using device: cuda
Slice Pair    | Users  | Recall   | NDCG     | EO       | DP       | Gini    
---------------------------------------------------------------------------------------------------------
1 -> 2        | 96     | 0.0178    | 0.0807    | 0.0065    | 0.0911    | 0.8636
PopExposure: {'mainstream': {'low': '20.85%', 'medium': '49.79%', 'high': '29.36%'}, 'niche': {'low': '33.27%', 'medium': '36.12%', 'high': '30.61%'}} W: {M:1.3/1.0/0.3, N:1.8/1.0/0.8}
2 -> 3        | 117    | 0.0263    | 0.0788    | 0.0045    | 0.1772    | 0.8302
PopExposure: {'mainstream': {'low': '15.58%', 'medium': '30.47%', 'high': '53.95%'}, 'niche': {'low': '42.16%', 'medium': '23.38%', 'high': '34.46%'}} W: {M:1.9/1.0/0.4, N:1.6/1.0/1.2}
3 -> 4        | 166    | 0.0267    | 0.0631    | 0.0284    | 0.1658    | 0.7898
PopExposure: {'mainstream': {'low': '19.84%', 'medium': '28.23%', 'high': '51.94%'}, 'niche': {'low': '44.71%', 'medium': '17.69%', 'high': '37.60%'}} W: {M:1.5/1.0/0.4, N:1.6/1.0/1.1}
4 

K = 25

In [21]:
import os
# CRITICAL FIX: Deterministic behavior for reproducibility
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import scipy.sparse as sp

# ----------------------------
# 1. Device & Setup
# ----------------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
np.random.seed(42)
torch.use_deterministic_algorithms(True)

# ----------------------------
# 2. Metric Helpers (Unchanged)
# ----------------------------
def get_gini(recommendation_counts):
    arr = np.sort(np.array(recommendation_counts))
    if np.sum(arr) == 0: return 0.0
    n = len(arr)
    return (n + 1 - 2 * np.sum(np.cumsum(arr)) / np.sum(arr)) / n

def compute_metrics(model, adj_dict, test_df, train_df, user_map, item_map, 
                    user_labels, movie_tiers, K=25):
    model.eval()
    with torch.no_grad():
        # KGCN Forward Pass
        final_emb = model(adj_dict)
        
        u_nodes = list(user_map.values())
        i_nodes = list(item_map.values())
        
        valid_test = test_df[test_df['UserID'].isin(user_map.keys()) & 
                             test_df['MovieID'].isin(item_map.keys())].copy()
        if valid_test.empty: return None, 0

        valid_test['u_node'] = valid_test['UserID'].map(user_map)
        valid_test['i_node'] = valid_test['MovieID'].map(item_map)
        
        user_gt = valid_test.groupby('u_node')['i_node'].apply(set).to_dict()
        test_u_nodes = list(user_gt.keys())
        
        test_u_tensor = torch.LongTensor(test_u_nodes).to(device)
        all_i_tensor = torch.LongTensor(i_nodes).to(device)
        
        user_embs = final_emb[test_u_tensor]
        item_embs = final_emb[all_i_tensor]
        
        scores = torch.matmul(user_embs, item_embs.t())
        
        train_interactions = train_df.groupby('UserID')['MovieID'].apply(list).to_dict()
        local_idx_to_node_id = {idx: node_id for idx, node_id in enumerate(i_nodes)}
        node_to_u_id_map = {v: k for k, v in user_map.items()}
        
        for idx, u_node in enumerate(test_u_nodes):
            u_id = node_to_u_id_map[u_node]
            if u_id in train_interactions:
                raw_seen = train_interactions[u_id]
                seen_nodes = set(item_map[m] for m in raw_seen if m in item_map)
                mask_indices = [k for k, v in local_idx_to_node_id.items() if v in seen_nodes]
                if mask_indices:
                    scores[idx, mask_indices] = -1e9

        _, top_indices = torch.topk(scores, K, dim=1)
        top_indices = top_indices.cpu().numpy()
        
        recall_arr, ndcg_arr = [], []
        group_recalls = {'mainstream': [], 'niche': []}
        pop_counts = {'mainstream': {'high': 0, 'medium': 0, 'low': 0}, 
                      'niche': {'high': 0, 'medium': 0, 'low': 0}}
        total_recs = {'mainstream': 0, 'niche': 0}
        
        node_to_m_id = {v: k for k, v in item_map.items()}
        all_item_counts = np.zeros(len(i_nodes))

        for idx, u_node in enumerate(test_u_nodes):
            truth = user_gt[u_node]
            rec_node_ids = [local_idx_to_node_id[x] for x in top_indices[idx]]
            
            hits = len(set(rec_node_ids) & truth)
            recall_arr.append(hits / len(truth))
            
            dcg, idcg = 0.0, 0.0
            for k, r in enumerate(rec_node_ids):
                if r in truth: dcg += 1.0 / np.log2(k + 2)
            for k in range(min(len(truth), K)): idcg += 1.0 / np.log2(k + 2)
            ndcg_arr.append(dcg/idcg if idcg > 0 else 0)
            
            u_id = node_to_u_id_map[u_node]
            u_type = user_labels.get(u_id, 'niche')
            group_recalls[u_type].append(hits / len(truth))
            total_recs[u_type] += K
            
            for local_idx, r_node in zip(top_indices[idx], rec_node_ids):
                all_item_counts[local_idx] += 1
                raw_mid = node_to_m_id.get(r_node)
                tier = movie_tiers.get(raw_mid, 'low')
                pop_counts[u_type][tier] += 1
        
        raw_exp = {grp: {t: (pop_counts[grp][t]/total_recs[grp]) if total_recs[grp]>0 else 0.0 for t in ['low','medium','high']} for grp in ['mainstream','niche']}
        formatted_pop = {grp: {t: f"{(raw_exp[grp][t]*100):.2f}%" for t in ['low','medium','high']} for grp in ['mainstream','niche']}
        
        eo = abs(np.mean(group_recalls['mainstream']) - np.mean(group_recalls['niche'])) if group_recalls['mainstream'] else 0
        dp = (abs(raw_exp['mainstream']['low'] - raw_exp['niche']['low']) +
              abs(raw_exp['mainstream']['medium'] - raw_exp['niche']['medium']) +
              abs(raw_exp['mainstream']['high'] - raw_exp['niche']['high'])) / 3.0
        gini_val = get_gini(all_item_counts)
        
        return {
            'Recall': np.mean(recall_arr), 'NDCG': np.mean(ndcg_arr), 'EO': eo, 'DP': dp, 
            'Gini': gini_val, 'PopExposure': formatted_pop, 'RawExposure': raw_exp
        }, len(test_u_nodes)

# ----------------------------
# 3. Model Logic: True KGCN (R-GCN Style)
# ----------------------------
class RGCNLayer(nn.Module):
    """
    True Relation-Aware Layer.
    Uses a specific Weight Matrix (W_r) for each relation type.
    Equation: h_u = Tanh( W_0 * h_u + Sum_r( Sum_v( W_r * h_v ) ) )
    This is 'fat' because it has params: num_relations * dim * dim.
    """
    def __init__(self, in_dim, out_dim, num_rels):
        super().__init__()
        self.in_dim = in_dim
        self.out_dim = out_dim
        self.num_rels = num_rels
        
        # The 'Fat' Part: A unique weight matrix for every relation type
        # We use nn.ParameterTensor for efficient lookup
        self.weight = nn.Parameter(torch.Tensor(num_rels, in_dim, out_dim))
        
        # Self-loop weight (how much the node keeps its own identity)
        self.w_self = nn.Linear(in_dim, out_dim)
        
        # Initialization
        nn.init.xavier_uniform_(self.weight)

    def forward(self, x, adj_dict):
        # x: [num_nodes, in_dim]
        
        # 1. Start with Self-Loop info
        out = self.w_self(x)
        
        # 2. Iterate over relations (Relation-Aware Aggregation)
        #  
        for rel_type, adj in adj_dict.items():
            # Safety check for relation index
            if rel_type >= self.num_rels: continue 
            
            # A. Retrieve the specific Weight Matrix for this relation
            w_r = self.weight[rel_type] # [in_dim, out_dim]
            
            # B. Transform node features specifically for this relation
            #    (Dense Matrix Mul -> GPU Efficient)
            x_transformed = torch.matmul(x, w_r) # [num_nodes, out_dim]
            
            # C. Propagate using the sparse adjacency
            #    (Sparse Matrix Mul)
            #    Aggregates the transformed neighbors
            out_r = torch.sparse.mm(adj, x_transformed)
            
            # Add to accumulator
            out = out + out_r
            
        return torch.tanh(out)

class TrueKGCN(nn.Module):
    def __init__(self, num_nodes, num_relations, embedding_dim=64, n_layers=2):
        super().__init__()
        self.node_emb = nn.Embedding(num_nodes, embedding_dim)
        
        # Stack of RGCN Layers
        self.layers = nn.ModuleList([
            RGCNLayer(embedding_dim, embedding_dim, num_relations)
            for _ in range(n_layers)
        ])
        
        nn.init.xavier_uniform_(self.node_emb.weight)

    def forward(self, adj_dict):
        x = self.node_emb.weight
        for layer in self.layers:
            x = layer(x, adj_dict)
        return x

def build_relational_adjacency(kg_edges_df, num_nodes):
    # Same logic, just builds the sparse matrices
    adj_dict = {}
    rel_types = kg_edges_df['RelType'].unique()
    
    for r in rel_types:
        sub_df = kg_edges_df[kg_edges_df['RelType'] == r]
        src = sub_df['Src'].values
        dst = sub_df['Dst'].values
        indices = np.vstack((src, dst))
        values = np.ones(len(src))
        
        i = torch.LongTensor(indices)
        v = torch.FloatTensor(values)
        shape = (num_nodes, num_nodes)
        
        adj = torch.sparse_coo_tensor(i, v, shape).coalesce()
        
        # Row-Normalize
        deg = torch.sparse.sum(adj, dim=1).to_dense()
        deg_inv = deg.pow(-1)
        deg_inv[deg_inv == float('inf')] = 0
        
        adj = torch.sparse_coo_tensor(i, deg_inv[i[0]], shape).to(device)
        adj_dict[r] = adj
        
    return adj_dict

# ----------------------------
# 4. Feedback Training Loop (Unchanged Workflow)
# ----------------------------
LR, EPOCHS, BATCH_SIZE = 0.001, 50, 2048
MAX_RETRIES = 10 

MAIN_TARGET_HIGH = (0.45, 0.55)
MAIN_TARGET_LOW  = (0.15, 0.25)
NICHE_TARGET_HIGH = (0.35, 0.45)
NICHE_TARGET_LOW  = (0.35, 0.45)

w_main_vals = [1.5, 1.0, 0.1]
w_niche_vals = [2.0, 1.0, 0.3]

prev_model_state = None
prev_node_map = {} 

print(f"{'Slice Pair':<13} | {'Users':<6} | {'Recall':<8} | {'NDCG':<8} | {'EO':<8} | {'DP':<8} | {'Gini':<8}")
print("-" * 105)

slice_nums = sorted(ratings['SliceNum'].unique())

for s in slice_nums[:-1]: 
    train_df = ratings[ratings['SliceNum'] == s].copy()
    test_df  = ratings[ratings['SliceNum'] == s+1].copy()
    
    user_map_df = pd.read_csv(f"slice_metadata/kg/user_map_slice_{s}.csv")
    movie_map_df = pd.read_csv(f"slice_metadata/kg/movie_map_slice_{s}.csv")
    kg_edges_df = pd.read_csv(f"slice_metadata/kg/kg_edges_slice_{s}.csv")
    
    u_map = dict(zip(user_map_df.UserID, user_map_df.NodeID))
    i_map = dict(zip(movie_map_df.MovieID, movie_map_df.NodeID))
    
    current_node_lookup = {}
    for k, v in u_map.items(): current_node_lookup[f"u_{k}"] = v
    for k, v in i_map.items(): current_node_lookup[f"m_{k}"] = v

    user_labels = pd.read_csv(f"slice_metadata/user_labels_slice_{s}.csv").set_index('UserID')['UserType'].to_dict()
    movie_tiers = pd.read_csv(f"slice_metadata/movie_pop_tiers_slice_{s}.csv").set_index('MovieID')['PopTier'].to_dict()

    max_node_id = max(kg_edges_df['Src'].max(), kg_edges_df['Dst'].max()) + 1
    
    # CRITICAL: Determine max relation ID for weight matrix dimensioning
    max_rel_id = kg_edges_df['RelType'].max() + 1
    
    adj_dict = build_relational_adjacency(kg_edges_df, max_node_id)
    
    train_df = train_df[train_df['UserID'].isin(u_map) & train_df['MovieID'].isin(i_map)]
    train_user_tensor = torch.LongTensor(train_df['UserID'].map(u_map).values).to(device)
    train_item_tensor = torch.LongTensor(train_df['MovieID'].map(i_map).values).to(device)
    num_train_samples = len(train_user_tensor)

    tier_idx_map = {'low': 0, 'medium': 1, 'high': 2}
    u_type_idx_map = {'mainstream': 0, 'niche': 1}
    
    item_tier_tensor = torch.zeros(max_node_id, dtype=torch.long, device=device)
    for mid, node_id in i_map.items():
        tier = movie_tiers.get(mid, 'low')
        item_tier_tensor[node_id] = tier_idx_map[tier]

    user_type_tensor = torch.zeros(max_node_id, dtype=torch.long, device=device)
    for uid, node_id in u_map.items():
        ut = user_labels.get(uid, 'niche')
        user_type_tensor[node_id] = u_type_idx_map[ut]

    retry_count = 1
    success = False
    
    while retry_count <= MAX_RETRIES and not success:
        # INSTANTIATE TRUE KGCN
        model = TrueKGCN(max_node_id, max_rel_id).to(device)
        
        if prev_model_state:
            with torch.no_grad():
                # Load Layer Weights (The 'Fat' parts)
                # We iterate layers to load W_r safely
                for i, layer in enumerate(model.layers):
                    w_name = f'layers.{i}.weight'
                    ws_name = f'layers.{i}.w_self.weight'
                    bs_name = f'layers.{i}.w_self.bias'
                    
                    if w_name in prev_model_state:
                        # Slice to fit in case num_rels changed (unlikely but safe)
                        prev_w = prev_model_state[w_name]
                        curr_rels = layer.weight.shape[0]
                        prev_rels = prev_w.shape[0]
                        sz = min(curr_rels, prev_rels)
                        layer.weight.data[:sz] = prev_w[:sz].to(device)
                        
                    if ws_name in prev_model_state:
                        layer.w_self.weight.data = prev_model_state[ws_name].to(device)
                        layer.w_self.bias.data = prev_model_state[bs_name].to(device)

                # Load node embeddings
                prev_nodes = prev_model_state['node_emb.weight'].to(device)
                for entity_key, curr_id in current_node_lookup.items():
                    if entity_key in prev_node_map:
                        prev_id = prev_node_map[entity_key]
                        if prev_id < prev_nodes.shape[0]:
                            model.node_emb.weight.data[curr_id] = prev_nodes[prev_id]

        optimizer = optim.Adam(model.parameters(), lr=LR)
        weight_matrix = torch.tensor([w_main_vals, w_niche_vals], dtype=torch.float, device=device)
        
        model.train()
        
        # Optimization: Move negative sampling pool to GPU
        all_item_nodes = torch.LongTensor(list(i_map.values())).to(device)
        
        for epoch in range(EPOCHS):
            rand_indices = torch.randint(0, len(all_item_nodes), (num_train_samples,), device=device)
            neg_nodes = all_item_nodes[rand_indices]
            
            idxs = torch.randperm(num_train_samples, device=device)
            
            for i in range(0, num_train_samples, BATCH_SIZE):
                b = idxs[i:i+BATCH_SIZE]
                
                u_b = train_user_tensor[b]
                p_b = train_item_tensor[b]
                n_b = neg_nodes[b]
                
                final_emb = model(adj_dict)
                
                pos_scores = (final_emb[u_b] * final_emb[p_b]).sum(1)
                neg_scores = (final_emb[u_b] * final_emb[n_b]).sum(1)
                bpr = -torch.nn.functional.logsigmoid(pos_scores - neg_scores)
                
                w_p = weight_matrix[user_type_tensor[u_b], item_tier_tensor[p_b]]
                loss = (bpr * w_p).mean()
                
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
        
        metrics, count = compute_metrics(
            model, adj_dict, test_df, train_df, 
            u_map, i_map, user_labels, movie_tiers
        )
        
        if metrics:
            m_h = metrics['RawExposure']['mainstream']['high']
            m_l = metrics['RawExposure']['mainstream']['low']
            n_h = metrics['RawExposure']['niche']['high']
            n_l = metrics['RawExposure']['niche']['low']
            
            needs_retry = False
            if m_h < MAIN_TARGET_HIGH[0]: w_main_vals[2] += 0.05; needs_retry = True
            elif m_h > MAIN_TARGET_HIGH[1]: w_main_vals[2] = max(0, w_main_vals[2] - 0.05); needs_retry = True
            if m_l < MAIN_TARGET_LOW[0]: w_main_vals[0] += 0.2; needs_retry = True
            elif m_l > MAIN_TARGET_LOW[1]: w_main_vals[0] = max(0, w_main_vals[0] - 0.2); needs_retry = True
            if n_h < NICHE_TARGET_HIGH[0]: w_niche_vals[2] += 0.05; needs_retry = True
            elif n_h > NICHE_TARGET_HIGH[1]: w_niche_vals[2] = max(0, w_niche_vals[2] - 0.05); needs_retry = True
            if n_l < NICHE_TARGET_LOW[0]: w_niche_vals[0] += 0.2; needs_retry = True
            elif n_l > NICHE_TARGET_LOW[1]: w_niche_vals[0] = max(0, w_niche_vals[0] - 0.2); needs_retry = True

            if not needs_retry or retry_count == MAX_RETRIES:
                print(f"{s} -> {s+1:<8} | {count:<6} | {metrics['Recall']:.4f}    | {metrics['NDCG']:.4f}    | {metrics['EO']:.4f}    | {metrics['DP']:.4f}    | {metrics['Gini']:.4f}")
                w_m_str = "/".join([f"{v:.1f}" for v in w_main_vals])
                w_n_str = "/".join([f"{v:.1f}" for v in w_niche_vals])
                print(f"PopExposure: {metrics['PopExposure']} W: {{M:{w_m_str}, N:{w_n_str}}}")
                
                success = True
                prev_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                prev_node_map = current_node_lookup.copy()
            else:
                retry_count += 1
        else:
            break

Using device: cuda
Slice Pair    | Users  | Recall   | NDCG     | EO       | DP       | Gini    
---------------------------------------------------------------------------------------------------------
1 -> 2        | 96     | 0.0423    | 0.0817    | 0.0146    | 0.1054    | 0.7837
PopExposure: {'mainstream': {'low': '22.47%', 'medium': '32.26%', 'high': '45.28%'}, 'niche': {'low': '37.96%', 'medium': '32.57%', 'high': '29.47%'}} W: {M:1.5/1.0/0.3, N:1.8/1.0/0.8}
2 -> 3        | 117    | 0.0453    | 0.0736    | 0.0247    | 0.1774    | 0.7172
PopExposure: {'mainstream': {'low': '14.79%', 'medium': '33.49%', 'high': '51.72%'}, 'niche': {'low': '41.41%', 'medium': '23.08%', 'high': '35.51%'}} W: {M:2.3/1.0/0.3, N:1.8/1.0/1.1}
3 -> 4        | 166    | 0.0476    | 0.0718    | 0.0375    | 0.1374    | 0.7033
PopExposure: {'mainstream': {'low': '21.74%', 'medium': '26.65%', 'high': '51.61%'}, 'niche': {'low': '42.35%', 'medium': '17.31%', 'high': '40.35%'}} W: {M:1.7/1.0/0.4, N:1.8/1.0/1.1}
4 

K = 50

In [22]:
import os
# CRITICAL FIX: Deterministic behavior for reproducibility
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import scipy.sparse as sp

# ----------------------------
# 1. Device & Setup
# ----------------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
np.random.seed(42)
torch.use_deterministic_algorithms(True)

# ----------------------------
# 2. Metric Helpers (Unchanged)
# ----------------------------
def get_gini(recommendation_counts):
    arr = np.sort(np.array(recommendation_counts))
    if np.sum(arr) == 0: return 0.0
    n = len(arr)
    return (n + 1 - 2 * np.sum(np.cumsum(arr)) / np.sum(arr)) / n

def compute_metrics(model, adj_dict, test_df, train_df, user_map, item_map, 
                    user_labels, movie_tiers, K=50):
    model.eval()
    with torch.no_grad():
        # KGCN Forward Pass
        final_emb = model(adj_dict)
        
        u_nodes = list(user_map.values())
        i_nodes = list(item_map.values())
        
        valid_test = test_df[test_df['UserID'].isin(user_map.keys()) & 
                             test_df['MovieID'].isin(item_map.keys())].copy()
        if valid_test.empty: return None, 0

        valid_test['u_node'] = valid_test['UserID'].map(user_map)
        valid_test['i_node'] = valid_test['MovieID'].map(item_map)
        
        user_gt = valid_test.groupby('u_node')['i_node'].apply(set).to_dict()
        test_u_nodes = list(user_gt.keys())
        
        test_u_tensor = torch.LongTensor(test_u_nodes).to(device)
        all_i_tensor = torch.LongTensor(i_nodes).to(device)
        
        user_embs = final_emb[test_u_tensor]
        item_embs = final_emb[all_i_tensor]
        
        scores = torch.matmul(user_embs, item_embs.t())
        
        train_interactions = train_df.groupby('UserID')['MovieID'].apply(list).to_dict()
        local_idx_to_node_id = {idx: node_id for idx, node_id in enumerate(i_nodes)}
        node_to_u_id_map = {v: k for k, v in user_map.items()}
        
        for idx, u_node in enumerate(test_u_nodes):
            u_id = node_to_u_id_map[u_node]
            if u_id in train_interactions:
                raw_seen = train_interactions[u_id]
                seen_nodes = set(item_map[m] for m in raw_seen if m in item_map)
                mask_indices = [k for k, v in local_idx_to_node_id.items() if v in seen_nodes]
                if mask_indices:
                    scores[idx, mask_indices] = -1e9

        _, top_indices = torch.topk(scores, K, dim=1)
        top_indices = top_indices.cpu().numpy()
        
        recall_arr, ndcg_arr = [], []
        group_recalls = {'mainstream': [], 'niche': []}
        pop_counts = {'mainstream': {'high': 0, 'medium': 0, 'low': 0}, 
                      'niche': {'high': 0, 'medium': 0, 'low': 0}}
        total_recs = {'mainstream': 0, 'niche': 0}
        
        node_to_m_id = {v: k for k, v in item_map.items()}
        all_item_counts = np.zeros(len(i_nodes))

        for idx, u_node in enumerate(test_u_nodes):
            truth = user_gt[u_node]
            rec_node_ids = [local_idx_to_node_id[x] for x in top_indices[idx]]
            
            hits = len(set(rec_node_ids) & truth)
            recall_arr.append(hits / len(truth))
            
            dcg, idcg = 0.0, 0.0
            for k, r in enumerate(rec_node_ids):
                if r in truth: dcg += 1.0 / np.log2(k + 2)
            for k in range(min(len(truth), K)): idcg += 1.0 / np.log2(k + 2)
            ndcg_arr.append(dcg/idcg if idcg > 0 else 0)
            
            u_id = node_to_u_id_map[u_node]
            u_type = user_labels.get(u_id, 'niche')
            group_recalls[u_type].append(hits / len(truth))
            total_recs[u_type] += K
            
            for local_idx, r_node in zip(top_indices[idx], rec_node_ids):
                all_item_counts[local_idx] += 1
                raw_mid = node_to_m_id.get(r_node)
                tier = movie_tiers.get(raw_mid, 'low')
                pop_counts[u_type][tier] += 1
        
        raw_exp = {grp: {t: (pop_counts[grp][t]/total_recs[grp]) if total_recs[grp]>0 else 0.0 for t in ['low','medium','high']} for grp in ['mainstream','niche']}
        formatted_pop = {grp: {t: f"{(raw_exp[grp][t]*100):.2f}%" for t in ['low','medium','high']} for grp in ['mainstream','niche']}
        
        eo = abs(np.mean(group_recalls['mainstream']) - np.mean(group_recalls['niche'])) if group_recalls['mainstream'] else 0
        dp = (abs(raw_exp['mainstream']['low'] - raw_exp['niche']['low']) +
              abs(raw_exp['mainstream']['medium'] - raw_exp['niche']['medium']) +
              abs(raw_exp['mainstream']['high'] - raw_exp['niche']['high'])) / 3.0
        gini_val = get_gini(all_item_counts)
        
        return {
            'Recall': np.mean(recall_arr), 'NDCG': np.mean(ndcg_arr), 'EO': eo, 'DP': dp, 
            'Gini': gini_val, 'PopExposure': formatted_pop, 'RawExposure': raw_exp
        }, len(test_u_nodes)

# ----------------------------
# 3. Model Logic: True KGCN (R-GCN Style)
# ----------------------------
class RGCNLayer(nn.Module):
    """
    True Relation-Aware Layer.
    Uses a specific Weight Matrix (W_r) for each relation type.
    Equation: h_u = Tanh( W_0 * h_u + Sum_r( Sum_v( W_r * h_v ) ) )
    This is 'fat' because it has params: num_relations * dim * dim.
    """
    def __init__(self, in_dim, out_dim, num_rels):
        super().__init__()
        self.in_dim = in_dim
        self.out_dim = out_dim
        self.num_rels = num_rels
        
        # The 'Fat' Part: A unique weight matrix for every relation type
        # We use nn.ParameterTensor for efficient lookup
        self.weight = nn.Parameter(torch.Tensor(num_rels, in_dim, out_dim))
        
        # Self-loop weight (how much the node keeps its own identity)
        self.w_self = nn.Linear(in_dim, out_dim)
        
        # Initialization
        nn.init.xavier_uniform_(self.weight)

    def forward(self, x, adj_dict):
        # x: [num_nodes, in_dim]
        
        # 1. Start with Self-Loop info
        out = self.w_self(x)
        
        # 2. Iterate over relations (Relation-Aware Aggregation)
        #  
        for rel_type, adj in adj_dict.items():
            # Safety check for relation index
            if rel_type >= self.num_rels: continue 
            
            # A. Retrieve the specific Weight Matrix for this relation
            w_r = self.weight[rel_type] # [in_dim, out_dim]
            
            # B. Transform node features specifically for this relation
            #    (Dense Matrix Mul -> GPU Efficient)
            x_transformed = torch.matmul(x, w_r) # [num_nodes, out_dim]
            
            # C. Propagate using the sparse adjacency
            #    (Sparse Matrix Mul)
            #    Aggregates the transformed neighbors
            out_r = torch.sparse.mm(adj, x_transformed)
            
            # Add to accumulator
            out = out + out_r
            
        return torch.tanh(out)

class TrueKGCN(nn.Module):
    def __init__(self, num_nodes, num_relations, embedding_dim=64, n_layers=2):
        super().__init__()
        self.node_emb = nn.Embedding(num_nodes, embedding_dim)
        
        # Stack of RGCN Layers
        self.layers = nn.ModuleList([
            RGCNLayer(embedding_dim, embedding_dim, num_relations)
            for _ in range(n_layers)
        ])
        
        nn.init.xavier_uniform_(self.node_emb.weight)

    def forward(self, adj_dict):
        x = self.node_emb.weight
        for layer in self.layers:
            x = layer(x, adj_dict)
        return x

def build_relational_adjacency(kg_edges_df, num_nodes):
    # Same logic, just builds the sparse matrices
    adj_dict = {}
    rel_types = kg_edges_df['RelType'].unique()
    
    for r in rel_types:
        sub_df = kg_edges_df[kg_edges_df['RelType'] == r]
        src = sub_df['Src'].values
        dst = sub_df['Dst'].values
        indices = np.vstack((src, dst))
        values = np.ones(len(src))
        
        i = torch.LongTensor(indices)
        v = torch.FloatTensor(values)
        shape = (num_nodes, num_nodes)
        
        adj = torch.sparse_coo_tensor(i, v, shape).coalesce()
        
        # Row-Normalize
        deg = torch.sparse.sum(adj, dim=1).to_dense()
        deg_inv = deg.pow(-1)
        deg_inv[deg_inv == float('inf')] = 0
        
        adj = torch.sparse_coo_tensor(i, deg_inv[i[0]], shape).to(device)
        adj_dict[r] = adj
        
    return adj_dict

# ----------------------------
# 4. Feedback Training Loop (Unchanged Workflow)
# ----------------------------
LR, EPOCHS, BATCH_SIZE = 0.001, 50, 2048
MAX_RETRIES = 10 

MAIN_TARGET_HIGH = (0.45, 0.55)
MAIN_TARGET_LOW  = (0.15, 0.25)
NICHE_TARGET_HIGH = (0.35, 0.45)
NICHE_TARGET_LOW  = (0.35, 0.45)

w_main_vals = [1.5, 1.0, 0.1]
w_niche_vals = [2.0, 1.0, 0.3]

prev_model_state = None
prev_node_map = {} 

print(f"{'Slice Pair':<13} | {'Users':<6} | {'Recall':<8} | {'NDCG':<8} | {'EO':<8} | {'DP':<8} | {'Gini':<8}")
print("-" * 105)

slice_nums = sorted(ratings['SliceNum'].unique())

for s in slice_nums[:-1]: 
    train_df = ratings[ratings['SliceNum'] == s].copy()
    test_df  = ratings[ratings['SliceNum'] == s+1].copy()
    
    user_map_df = pd.read_csv(f"slice_metadata/kg/user_map_slice_{s}.csv")
    movie_map_df = pd.read_csv(f"slice_metadata/kg/movie_map_slice_{s}.csv")
    kg_edges_df = pd.read_csv(f"slice_metadata/kg/kg_edges_slice_{s}.csv")
    
    u_map = dict(zip(user_map_df.UserID, user_map_df.NodeID))
    i_map = dict(zip(movie_map_df.MovieID, movie_map_df.NodeID))
    
    current_node_lookup = {}
    for k, v in u_map.items(): current_node_lookup[f"u_{k}"] = v
    for k, v in i_map.items(): current_node_lookup[f"m_{k}"] = v

    user_labels = pd.read_csv(f"slice_metadata/user_labels_slice_{s}.csv").set_index('UserID')['UserType'].to_dict()
    movie_tiers = pd.read_csv(f"slice_metadata/movie_pop_tiers_slice_{s}.csv").set_index('MovieID')['PopTier'].to_dict()

    max_node_id = max(kg_edges_df['Src'].max(), kg_edges_df['Dst'].max()) + 1
    
    # CRITICAL: Determine max relation ID for weight matrix dimensioning
    max_rel_id = kg_edges_df['RelType'].max() + 1
    
    adj_dict = build_relational_adjacency(kg_edges_df, max_node_id)
    
    train_df = train_df[train_df['UserID'].isin(u_map) & train_df['MovieID'].isin(i_map)]
    train_user_tensor = torch.LongTensor(train_df['UserID'].map(u_map).values).to(device)
    train_item_tensor = torch.LongTensor(train_df['MovieID'].map(i_map).values).to(device)
    num_train_samples = len(train_user_tensor)

    tier_idx_map = {'low': 0, 'medium': 1, 'high': 2}
    u_type_idx_map = {'mainstream': 0, 'niche': 1}
    
    item_tier_tensor = torch.zeros(max_node_id, dtype=torch.long, device=device)
    for mid, node_id in i_map.items():
        tier = movie_tiers.get(mid, 'low')
        item_tier_tensor[node_id] = tier_idx_map[tier]

    user_type_tensor = torch.zeros(max_node_id, dtype=torch.long, device=device)
    for uid, node_id in u_map.items():
        ut = user_labels.get(uid, 'niche')
        user_type_tensor[node_id] = u_type_idx_map[ut]

    retry_count = 1
    success = False
    
    while retry_count <= MAX_RETRIES and not success:
        # INSTANTIATE TRUE KGCN
        model = TrueKGCN(max_node_id, max_rel_id).to(device)
        
        if prev_model_state:
            with torch.no_grad():
                # Load Layer Weights (The 'Fat' parts)
                # We iterate layers to load W_r safely
                for i, layer in enumerate(model.layers):
                    w_name = f'layers.{i}.weight'
                    ws_name = f'layers.{i}.w_self.weight'
                    bs_name = f'layers.{i}.w_self.bias'
                    
                    if w_name in prev_model_state:
                        # Slice to fit in case num_rels changed (unlikely but safe)
                        prev_w = prev_model_state[w_name]
                        curr_rels = layer.weight.shape[0]
                        prev_rels = prev_w.shape[0]
                        sz = min(curr_rels, prev_rels)
                        layer.weight.data[:sz] = prev_w[:sz].to(device)
                        
                    if ws_name in prev_model_state:
                        layer.w_self.weight.data = prev_model_state[ws_name].to(device)
                        layer.w_self.bias.data = prev_model_state[bs_name].to(device)

                # Load node embeddings
                prev_nodes = prev_model_state['node_emb.weight'].to(device)
                for entity_key, curr_id in current_node_lookup.items():
                    if entity_key in prev_node_map:
                        prev_id = prev_node_map[entity_key]
                        if prev_id < prev_nodes.shape[0]:
                            model.node_emb.weight.data[curr_id] = prev_nodes[prev_id]

        optimizer = optim.Adam(model.parameters(), lr=LR)
        weight_matrix = torch.tensor([w_main_vals, w_niche_vals], dtype=torch.float, device=device)
        
        model.train()
        
        # Optimization: Move negative sampling pool to GPU
        all_item_nodes = torch.LongTensor(list(i_map.values())).to(device)
        
        for epoch in range(EPOCHS):
            rand_indices = torch.randint(0, len(all_item_nodes), (num_train_samples,), device=device)
            neg_nodes = all_item_nodes[rand_indices]
            
            idxs = torch.randperm(num_train_samples, device=device)
            
            for i in range(0, num_train_samples, BATCH_SIZE):
                b = idxs[i:i+BATCH_SIZE]
                
                u_b = train_user_tensor[b]
                p_b = train_item_tensor[b]
                n_b = neg_nodes[b]
                
                final_emb = model(adj_dict)
                
                pos_scores = (final_emb[u_b] * final_emb[p_b]).sum(1)
                neg_scores = (final_emb[u_b] * final_emb[n_b]).sum(1)
                bpr = -torch.nn.functional.logsigmoid(pos_scores - neg_scores)
                
                w_p = weight_matrix[user_type_tensor[u_b], item_tier_tensor[p_b]]
                loss = (bpr * w_p).mean()
                
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
        
        metrics, count = compute_metrics(
            model, adj_dict, test_df, train_df, 
            u_map, i_map, user_labels, movie_tiers
        )
        
        if metrics:
            m_h = metrics['RawExposure']['mainstream']['high']
            m_l = metrics['RawExposure']['mainstream']['low']
            n_h = metrics['RawExposure']['niche']['high']
            n_l = metrics['RawExposure']['niche']['low']
            
            needs_retry = False
            if m_h < MAIN_TARGET_HIGH[0]: w_main_vals[2] += 0.05; needs_retry = True
            elif m_h > MAIN_TARGET_HIGH[1]: w_main_vals[2] = max(0, w_main_vals[2] - 0.05); needs_retry = True
            if m_l < MAIN_TARGET_LOW[0]: w_main_vals[0] += 0.2; needs_retry = True
            elif m_l > MAIN_TARGET_LOW[1]: w_main_vals[0] = max(0, w_main_vals[0] - 0.2); needs_retry = True
            if n_h < NICHE_TARGET_HIGH[0]: w_niche_vals[2] += 0.05; needs_retry = True
            elif n_h > NICHE_TARGET_HIGH[1]: w_niche_vals[2] = max(0, w_niche_vals[2] - 0.05); needs_retry = True
            if n_l < NICHE_TARGET_LOW[0]: w_niche_vals[0] += 0.2; needs_retry = True
            elif n_l > NICHE_TARGET_LOW[1]: w_niche_vals[0] = max(0, w_niche_vals[0] - 0.2); needs_retry = True

            if not needs_retry or retry_count == MAX_RETRIES:
                print(f"{s} -> {s+1:<8} | {count:<6} | {metrics['Recall']:.4f}    | {metrics['NDCG']:.4f}    | {metrics['EO']:.4f}    | {metrics['DP']:.4f}    | {metrics['Gini']:.4f}")
                w_m_str = "/".join([f"{v:.1f}" for v in w_main_vals])
                w_n_str = "/".join([f"{v:.1f}" for v in w_niche_vals])
                print(f"PopExposure: {metrics['PopExposure']} W: {{M:{w_m_str}, N:{w_n_str}}}")
                
                success = True
                prev_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                prev_node_map = current_node_lookup.copy()
            else:
                retry_count += 1
        else:
            break

Using device: cuda
Slice Pair    | Users  | Recall   | NDCG     | EO       | DP       | Gini    
---------------------------------------------------------------------------------------------------------
1 -> 2        | 96     | 0.0790    | 0.0862    | 0.0347    | 0.1125    | 0.6924
PopExposure: {'mainstream': {'low': '21.62%', 'medium': '37.91%', 'high': '40.47%'}, 'niche': {'low': '38.49%', 'medium': '32.37%', 'high': '29.14%'}} W: {M:1.5/1.0/0.3, N:2.0/1.0/0.8}
2 -> 3        | 117    | 0.0922    | 0.0793    | 0.0445    | 0.1346    | 0.6315
PopExposure: {'mainstream': {'low': '16.05%', 'medium': '35.95%', 'high': '48.00%'}, 'niche': {'low': '36.24%', 'medium': '25.08%', 'high': '38.68%'}} W: {M:2.3/1.0/0.3, N:2.0/1.0/1.1}
3 -> 4        | 166    | 0.1100    | 0.0801    | 0.0357    | 0.1154    | 0.6353
PopExposure: {'mainstream': {'low': '22.61%', 'medium': '29.19%', 'high': '48.19%'}, 'niche': {'low': '39.92%', 'medium': '18.44%', 'high': '41.63%'}} W: {M:2.1/1.0/0.4, N:2.0/1.0/1.1}
4 

K = 75

In [23]:
import os
# CRITICAL FIX: Deterministic behavior for reproducibility
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import scipy.sparse as sp

# ----------------------------
# 1. Device & Setup
# ----------------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
np.random.seed(42)
torch.use_deterministic_algorithms(True)

# ----------------------------
# 2. Metric Helpers (Unchanged)
# ----------------------------
def get_gini(recommendation_counts):
    arr = np.sort(np.array(recommendation_counts))
    if np.sum(arr) == 0: return 0.0
    n = len(arr)
    return (n + 1 - 2 * np.sum(np.cumsum(arr)) / np.sum(arr)) / n

def compute_metrics(model, adj_dict, test_df, train_df, user_map, item_map, 
                    user_labels, movie_tiers, K=75):
    model.eval()
    with torch.no_grad():
        # KGCN Forward Pass
        final_emb = model(adj_dict)
        
        u_nodes = list(user_map.values())
        i_nodes = list(item_map.values())
        
        valid_test = test_df[test_df['UserID'].isin(user_map.keys()) & 
                             test_df['MovieID'].isin(item_map.keys())].copy()
        if valid_test.empty: return None, 0

        valid_test['u_node'] = valid_test['UserID'].map(user_map)
        valid_test['i_node'] = valid_test['MovieID'].map(item_map)
        
        user_gt = valid_test.groupby('u_node')['i_node'].apply(set).to_dict()
        test_u_nodes = list(user_gt.keys())
        
        test_u_tensor = torch.LongTensor(test_u_nodes).to(device)
        all_i_tensor = torch.LongTensor(i_nodes).to(device)
        
        user_embs = final_emb[test_u_tensor]
        item_embs = final_emb[all_i_tensor]
        
        scores = torch.matmul(user_embs, item_embs.t())
        
        train_interactions = train_df.groupby('UserID')['MovieID'].apply(list).to_dict()
        local_idx_to_node_id = {idx: node_id for idx, node_id in enumerate(i_nodes)}
        node_to_u_id_map = {v: k for k, v in user_map.items()}
        
        for idx, u_node in enumerate(test_u_nodes):
            u_id = node_to_u_id_map[u_node]
            if u_id in train_interactions:
                raw_seen = train_interactions[u_id]
                seen_nodes = set(item_map[m] for m in raw_seen if m in item_map)
                mask_indices = [k for k, v in local_idx_to_node_id.items() if v in seen_nodes]
                if mask_indices:
                    scores[idx, mask_indices] = -1e9

        _, top_indices = torch.topk(scores, K, dim=1)
        top_indices = top_indices.cpu().numpy()
        
        recall_arr, ndcg_arr = [], []
        group_recalls = {'mainstream': [], 'niche': []}
        pop_counts = {'mainstream': {'high': 0, 'medium': 0, 'low': 0}, 
                      'niche': {'high': 0, 'medium': 0, 'low': 0}}
        total_recs = {'mainstream': 0, 'niche': 0}
        
        node_to_m_id = {v: k for k, v in item_map.items()}
        all_item_counts = np.zeros(len(i_nodes))

        for idx, u_node in enumerate(test_u_nodes):
            truth = user_gt[u_node]
            rec_node_ids = [local_idx_to_node_id[x] for x in top_indices[idx]]
            
            hits = len(set(rec_node_ids) & truth)
            recall_arr.append(hits / len(truth))
            
            dcg, idcg = 0.0, 0.0
            for k, r in enumerate(rec_node_ids):
                if r in truth: dcg += 1.0 / np.log2(k + 2)
            for k in range(min(len(truth), K)): idcg += 1.0 / np.log2(k + 2)
            ndcg_arr.append(dcg/idcg if idcg > 0 else 0)
            
            u_id = node_to_u_id_map[u_node]
            u_type = user_labels.get(u_id, 'niche')
            group_recalls[u_type].append(hits / len(truth))
            total_recs[u_type] += K
            
            for local_idx, r_node in zip(top_indices[idx], rec_node_ids):
                all_item_counts[local_idx] += 1
                raw_mid = node_to_m_id.get(r_node)
                tier = movie_tiers.get(raw_mid, 'low')
                pop_counts[u_type][tier] += 1
        
        raw_exp = {grp: {t: (pop_counts[grp][t]/total_recs[grp]) if total_recs[grp]>0 else 0.0 for t in ['low','medium','high']} for grp in ['mainstream','niche']}
        formatted_pop = {grp: {t: f"{(raw_exp[grp][t]*100):.2f}%" for t in ['low','medium','high']} for grp in ['mainstream','niche']}
        
        eo = abs(np.mean(group_recalls['mainstream']) - np.mean(group_recalls['niche'])) if group_recalls['mainstream'] else 0
        dp = (abs(raw_exp['mainstream']['low'] - raw_exp['niche']['low']) +
              abs(raw_exp['mainstream']['medium'] - raw_exp['niche']['medium']) +
              abs(raw_exp['mainstream']['high'] - raw_exp['niche']['high'])) / 3.0
        gini_val = get_gini(all_item_counts)
        
        return {
            'Recall': np.mean(recall_arr), 'NDCG': np.mean(ndcg_arr), 'EO': eo, 'DP': dp, 
            'Gini': gini_val, 'PopExposure': formatted_pop, 'RawExposure': raw_exp
        }, len(test_u_nodes)

# ----------------------------
# 3. Model Logic: True KGCN (R-GCN Style)
# ----------------------------
class RGCNLayer(nn.Module):
    """
    True Relation-Aware Layer.
    Uses a specific Weight Matrix (W_r) for each relation type.
    Equation: h_u = Tanh( W_0 * h_u + Sum_r( Sum_v( W_r * h_v ) ) )
    This is 'fat' because it has params: num_relations * dim * dim.
    """
    def __init__(self, in_dim, out_dim, num_rels):
        super().__init__()
        self.in_dim = in_dim
        self.out_dim = out_dim
        self.num_rels = num_rels
        
        # The 'Fat' Part: A unique weight matrix for every relation type
        # We use nn.ParameterTensor for efficient lookup
        self.weight = nn.Parameter(torch.Tensor(num_rels, in_dim, out_dim))
        
        # Self-loop weight (how much the node keeps its own identity)
        self.w_self = nn.Linear(in_dim, out_dim)
        
        # Initialization
        nn.init.xavier_uniform_(self.weight)

    def forward(self, x, adj_dict):
        # x: [num_nodes, in_dim]
        
        # 1. Start with Self-Loop info
        out = self.w_self(x)
        
        # 2. Iterate over relations (Relation-Aware Aggregation)
        #  
        for rel_type, adj in adj_dict.items():
            # Safety check for relation index
            if rel_type >= self.num_rels: continue 
            
            # A. Retrieve the specific Weight Matrix for this relation
            w_r = self.weight[rel_type] # [in_dim, out_dim]
            
            # B. Transform node features specifically for this relation
            #    (Dense Matrix Mul -> GPU Efficient)
            x_transformed = torch.matmul(x, w_r) # [num_nodes, out_dim]
            
            # C. Propagate using the sparse adjacency
            #    (Sparse Matrix Mul)
            #    Aggregates the transformed neighbors
            out_r = torch.sparse.mm(adj, x_transformed)
            
            # Add to accumulator
            out = out + out_r
            
        return torch.tanh(out)

class TrueKGCN(nn.Module):
    def __init__(self, num_nodes, num_relations, embedding_dim=64, n_layers=2):
        super().__init__()
        self.node_emb = nn.Embedding(num_nodes, embedding_dim)
        
        # Stack of RGCN Layers
        self.layers = nn.ModuleList([
            RGCNLayer(embedding_dim, embedding_dim, num_relations)
            for _ in range(n_layers)
        ])
        
        nn.init.xavier_uniform_(self.node_emb.weight)

    def forward(self, adj_dict):
        x = self.node_emb.weight
        for layer in self.layers:
            x = layer(x, adj_dict)
        return x

def build_relational_adjacency(kg_edges_df, num_nodes):
    # Same logic, just builds the sparse matrices
    adj_dict = {}
    rel_types = kg_edges_df['RelType'].unique()
    
    for r in rel_types:
        sub_df = kg_edges_df[kg_edges_df['RelType'] == r]
        src = sub_df['Src'].values
        dst = sub_df['Dst'].values
        indices = np.vstack((src, dst))
        values = np.ones(len(src))
        
        i = torch.LongTensor(indices)
        v = torch.FloatTensor(values)
        shape = (num_nodes, num_nodes)
        
        adj = torch.sparse_coo_tensor(i, v, shape).coalesce()
        
        # Row-Normalize
        deg = torch.sparse.sum(adj, dim=1).to_dense()
        deg_inv = deg.pow(-1)
        deg_inv[deg_inv == float('inf')] = 0
        
        adj = torch.sparse_coo_tensor(i, deg_inv[i[0]], shape).to(device)
        adj_dict[r] = adj
        
    return adj_dict

# ----------------------------
# 4. Feedback Training Loop (Unchanged Workflow)
# ----------------------------
LR, EPOCHS, BATCH_SIZE = 0.001, 50, 2048
MAX_RETRIES = 10 

MAIN_TARGET_HIGH = (0.45, 0.55)
MAIN_TARGET_LOW  = (0.15, 0.25)
NICHE_TARGET_HIGH = (0.35, 0.45)
NICHE_TARGET_LOW  = (0.35, 0.45)

w_main_vals = [1.5, 1.0, 0.1]
w_niche_vals = [2.0, 1.0, 0.3]

prev_model_state = None
prev_node_map = {} 

print(f"{'Slice Pair':<13} | {'Users':<6} | {'Recall':<8} | {'NDCG':<8} | {'EO':<8} | {'DP':<8} | {'Gini':<8}")
print("-" * 105)

slice_nums = sorted(ratings['SliceNum'].unique())

for s in slice_nums[:-1]: 
    train_df = ratings[ratings['SliceNum'] == s].copy()
    test_df  = ratings[ratings['SliceNum'] == s+1].copy()
    
    user_map_df = pd.read_csv(f"slice_metadata/kg/user_map_slice_{s}.csv")
    movie_map_df = pd.read_csv(f"slice_metadata/kg/movie_map_slice_{s}.csv")
    kg_edges_df = pd.read_csv(f"slice_metadata/kg/kg_edges_slice_{s}.csv")
    
    u_map = dict(zip(user_map_df.UserID, user_map_df.NodeID))
    i_map = dict(zip(movie_map_df.MovieID, movie_map_df.NodeID))
    
    current_node_lookup = {}
    for k, v in u_map.items(): current_node_lookup[f"u_{k}"] = v
    for k, v in i_map.items(): current_node_lookup[f"m_{k}"] = v

    user_labels = pd.read_csv(f"slice_metadata/user_labels_slice_{s}.csv").set_index('UserID')['UserType'].to_dict()
    movie_tiers = pd.read_csv(f"slice_metadata/movie_pop_tiers_slice_{s}.csv").set_index('MovieID')['PopTier'].to_dict()

    max_node_id = max(kg_edges_df['Src'].max(), kg_edges_df['Dst'].max()) + 1
    
    # CRITICAL: Determine max relation ID for weight matrix dimensioning
    max_rel_id = kg_edges_df['RelType'].max() + 1
    
    adj_dict = build_relational_adjacency(kg_edges_df, max_node_id)
    
    train_df = train_df[train_df['UserID'].isin(u_map) & train_df['MovieID'].isin(i_map)]
    train_user_tensor = torch.LongTensor(train_df['UserID'].map(u_map).values).to(device)
    train_item_tensor = torch.LongTensor(train_df['MovieID'].map(i_map).values).to(device)
    num_train_samples = len(train_user_tensor)

    tier_idx_map = {'low': 0, 'medium': 1, 'high': 2}
    u_type_idx_map = {'mainstream': 0, 'niche': 1}
    
    item_tier_tensor = torch.zeros(max_node_id, dtype=torch.long, device=device)
    for mid, node_id in i_map.items():
        tier = movie_tiers.get(mid, 'low')
        item_tier_tensor[node_id] = tier_idx_map[tier]

    user_type_tensor = torch.zeros(max_node_id, dtype=torch.long, device=device)
    for uid, node_id in u_map.items():
        ut = user_labels.get(uid, 'niche')
        user_type_tensor[node_id] = u_type_idx_map[ut]

    retry_count = 1
    success = False
    
    while retry_count <= MAX_RETRIES and not success:
        # INSTANTIATE TRUE KGCN
        model = TrueKGCN(max_node_id, max_rel_id).to(device)
        
        if prev_model_state:
            with torch.no_grad():
                # Load Layer Weights (The 'Fat' parts)
                # We iterate layers to load W_r safely
                for i, layer in enumerate(model.layers):
                    w_name = f'layers.{i}.weight'
                    ws_name = f'layers.{i}.w_self.weight'
                    bs_name = f'layers.{i}.w_self.bias'
                    
                    if w_name in prev_model_state:
                        # Slice to fit in case num_rels changed (unlikely but safe)
                        prev_w = prev_model_state[w_name]
                        curr_rels = layer.weight.shape[0]
                        prev_rels = prev_w.shape[0]
                        sz = min(curr_rels, prev_rels)
                        layer.weight.data[:sz] = prev_w[:sz].to(device)
                        
                    if ws_name in prev_model_state:
                        layer.w_self.weight.data = prev_model_state[ws_name].to(device)
                        layer.w_self.bias.data = prev_model_state[bs_name].to(device)

                # Load node embeddings
                prev_nodes = prev_model_state['node_emb.weight'].to(device)
                for entity_key, curr_id in current_node_lookup.items():
                    if entity_key in prev_node_map:
                        prev_id = prev_node_map[entity_key]
                        if prev_id < prev_nodes.shape[0]:
                            model.node_emb.weight.data[curr_id] = prev_nodes[prev_id]

        optimizer = optim.Adam(model.parameters(), lr=LR)
        weight_matrix = torch.tensor([w_main_vals, w_niche_vals], dtype=torch.float, device=device)
        
        model.train()
        
        # Optimization: Move negative sampling pool to GPU
        all_item_nodes = torch.LongTensor(list(i_map.values())).to(device)
        
        for epoch in range(EPOCHS):
            rand_indices = torch.randint(0, len(all_item_nodes), (num_train_samples,), device=device)
            neg_nodes = all_item_nodes[rand_indices]
            
            idxs = torch.randperm(num_train_samples, device=device)
            
            for i in range(0, num_train_samples, BATCH_SIZE):
                b = idxs[i:i+BATCH_SIZE]
                
                u_b = train_user_tensor[b]
                p_b = train_item_tensor[b]
                n_b = neg_nodes[b]
                
                final_emb = model(adj_dict)
                
                pos_scores = (final_emb[u_b] * final_emb[p_b]).sum(1)
                neg_scores = (final_emb[u_b] * final_emb[n_b]).sum(1)
                bpr = -torch.nn.functional.logsigmoid(pos_scores - neg_scores)
                
                w_p = weight_matrix[user_type_tensor[u_b], item_tier_tensor[p_b]]
                loss = (bpr * w_p).mean()
                
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
        
        metrics, count = compute_metrics(
            model, adj_dict, test_df, train_df, 
            u_map, i_map, user_labels, movie_tiers
        )
        
        if metrics:
            m_h = metrics['RawExposure']['mainstream']['high']
            m_l = metrics['RawExposure']['mainstream']['low']
            n_h = metrics['RawExposure']['niche']['high']
            n_l = metrics['RawExposure']['niche']['low']
            
            needs_retry = False
            if m_h < MAIN_TARGET_HIGH[0]: w_main_vals[2] += 0.05; needs_retry = True
            elif m_h > MAIN_TARGET_HIGH[1]: w_main_vals[2] = max(0, w_main_vals[2] - 0.05); needs_retry = True
            if m_l < MAIN_TARGET_LOW[0]: w_main_vals[0] += 0.2; needs_retry = True
            elif m_l > MAIN_TARGET_LOW[1]: w_main_vals[0] = max(0, w_main_vals[0] - 0.2); needs_retry = True
            if n_h < NICHE_TARGET_HIGH[0]: w_niche_vals[2] += 0.05; needs_retry = True
            elif n_h > NICHE_TARGET_HIGH[1]: w_niche_vals[2] = max(0, w_niche_vals[2] - 0.05); needs_retry = True
            if n_l < NICHE_TARGET_LOW[0]: w_niche_vals[0] += 0.2; needs_retry = True
            elif n_l > NICHE_TARGET_LOW[1]: w_niche_vals[0] = max(0, w_niche_vals[0] - 0.2); needs_retry = True

            if not needs_retry or retry_count == MAX_RETRIES:
                print(f"{s} -> {s+1:<8} | {count:<6} | {metrics['Recall']:.4f}    | {metrics['NDCG']:.4f}    | {metrics['EO']:.4f}    | {metrics['DP']:.4f}    | {metrics['Gini']:.4f}")
                w_m_str = "/".join([f"{v:.1f}" for v in w_main_vals])
                w_n_str = "/".join([f"{v:.1f}" for v in w_niche_vals])
                print(f"PopExposure: {metrics['PopExposure']} W: {{M:{w_m_str}, N:{w_n_str}}}")
                
                success = True
                prev_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                prev_node_map = current_node_lookup.copy()
            else:
                retry_count += 1
        else:
            break

Using device: cuda
Slice Pair    | Users  | Recall   | NDCG     | EO       | DP       | Gini    
---------------------------------------------------------------------------------------------------------
1 -> 2        | 96     | 0.1211    | 0.0984    | 0.0625    | 0.1258    | 0.6474
PopExposure: {'mainstream': {'low': '21.53%', 'medium': '34.44%', 'high': '44.03%'}, 'niche': {'low': '40.41%', 'medium': '29.52%', 'high': '30.07%'}} W: {M:1.7/1.0/0.3, N:2.2/1.0/0.8}
2 -> 3        | 117    | 0.1166    | 0.0923    | 0.0614    | 0.1296    | 0.5996
PopExposure: {'mainstream': {'low': '15.97%', 'medium': '30.39%', 'high': '53.64%'}, 'niche': {'low': '35.41%', 'medium': '26.56%', 'high': '38.04%'}} W: {M:2.3/1.0/0.3, N:2.2/1.0/1.0}
3 -> 4        | 166    | 0.1373    | 0.0816    | 0.0357    | 0.1134    | 0.5995
PopExposure: {'mainstream': {'low': '22.84%', 'medium': '28.43%', 'high': '48.73%'}, 'niche': {'low': '39.85%', 'medium': '21.42%', 'high': '38.73%'}} W: {M:2.3/1.0/0.4, N:2.2/1.0/1.0}
4 

K = 100

In [24]:
import os
# CRITICAL FIX: Deterministic behavior for reproducibility
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import scipy.sparse as sp

# ----------------------------
# 1. Device & Setup
# ----------------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
np.random.seed(42)
torch.use_deterministic_algorithms(True)

# ----------------------------
# 2. Metric Helpers (Unchanged)
# ----------------------------
def get_gini(recommendation_counts):
    arr = np.sort(np.array(recommendation_counts))
    if np.sum(arr) == 0: return 0.0
    n = len(arr)
    return (n + 1 - 2 * np.sum(np.cumsum(arr)) / np.sum(arr)) / n

def compute_metrics(model, adj_dict, test_df, train_df, user_map, item_map, 
                    user_labels, movie_tiers, K=100):
    model.eval()
    with torch.no_grad():
        # KGCN Forward Pass
        final_emb = model(adj_dict)
        
        u_nodes = list(user_map.values())
        i_nodes = list(item_map.values())
        
        valid_test = test_df[test_df['UserID'].isin(user_map.keys()) & 
                             test_df['MovieID'].isin(item_map.keys())].copy()
        if valid_test.empty: return None, 0

        valid_test['u_node'] = valid_test['UserID'].map(user_map)
        valid_test['i_node'] = valid_test['MovieID'].map(item_map)
        
        user_gt = valid_test.groupby('u_node')['i_node'].apply(set).to_dict()
        test_u_nodes = list(user_gt.keys())
        
        test_u_tensor = torch.LongTensor(test_u_nodes).to(device)
        all_i_tensor = torch.LongTensor(i_nodes).to(device)
        
        user_embs = final_emb[test_u_tensor]
        item_embs = final_emb[all_i_tensor]
        
        scores = torch.matmul(user_embs, item_embs.t())
        
        train_interactions = train_df.groupby('UserID')['MovieID'].apply(list).to_dict()
        local_idx_to_node_id = {idx: node_id for idx, node_id in enumerate(i_nodes)}
        node_to_u_id_map = {v: k for k, v in user_map.items()}
        
        for idx, u_node in enumerate(test_u_nodes):
            u_id = node_to_u_id_map[u_node]
            if u_id in train_interactions:
                raw_seen = train_interactions[u_id]
                seen_nodes = set(item_map[m] for m in raw_seen if m in item_map)
                mask_indices = [k for k, v in local_idx_to_node_id.items() if v in seen_nodes]
                if mask_indices:
                    scores[idx, mask_indices] = -1e9

        _, top_indices = torch.topk(scores, K, dim=1)
        top_indices = top_indices.cpu().numpy()
        
        recall_arr, ndcg_arr = [], []
        group_recalls = {'mainstream': [], 'niche': []}
        pop_counts = {'mainstream': {'high': 0, 'medium': 0, 'low': 0}, 
                      'niche': {'high': 0, 'medium': 0, 'low': 0}}
        total_recs = {'mainstream': 0, 'niche': 0}
        
        node_to_m_id = {v: k for k, v in item_map.items()}
        all_item_counts = np.zeros(len(i_nodes))

        for idx, u_node in enumerate(test_u_nodes):
            truth = user_gt[u_node]
            rec_node_ids = [local_idx_to_node_id[x] for x in top_indices[idx]]
            
            hits = len(set(rec_node_ids) & truth)
            recall_arr.append(hits / len(truth))
            
            dcg, idcg = 0.0, 0.0
            for k, r in enumerate(rec_node_ids):
                if r in truth: dcg += 1.0 / np.log2(k + 2)
            for k in range(min(len(truth), K)): idcg += 1.0 / np.log2(k + 2)
            ndcg_arr.append(dcg/idcg if idcg > 0 else 0)
            
            u_id = node_to_u_id_map[u_node]
            u_type = user_labels.get(u_id, 'niche')
            group_recalls[u_type].append(hits / len(truth))
            total_recs[u_type] += K
            
            for local_idx, r_node in zip(top_indices[idx], rec_node_ids):
                all_item_counts[local_idx] += 1
                raw_mid = node_to_m_id.get(r_node)
                tier = movie_tiers.get(raw_mid, 'low')
                pop_counts[u_type][tier] += 1
        
        raw_exp = {grp: {t: (pop_counts[grp][t]/total_recs[grp]) if total_recs[grp]>0 else 0.0 for t in ['low','medium','high']} for grp in ['mainstream','niche']}
        formatted_pop = {grp: {t: f"{(raw_exp[grp][t]*100):.2f}%" for t in ['low','medium','high']} for grp in ['mainstream','niche']}
        
        eo = abs(np.mean(group_recalls['mainstream']) - np.mean(group_recalls['niche'])) if group_recalls['mainstream'] else 0
        dp = (abs(raw_exp['mainstream']['low'] - raw_exp['niche']['low']) +
              abs(raw_exp['mainstream']['medium'] - raw_exp['niche']['medium']) +
              abs(raw_exp['mainstream']['high'] - raw_exp['niche']['high'])) / 3.0
        gini_val = get_gini(all_item_counts)
        
        return {
            'Recall': np.mean(recall_arr), 'NDCG': np.mean(ndcg_arr), 'EO': eo, 'DP': dp, 
            'Gini': gini_val, 'PopExposure': formatted_pop, 'RawExposure': raw_exp
        }, len(test_u_nodes)

# ----------------------------
# 3. Model Logic: True KGCN (R-GCN Style)
# ----------------------------
class RGCNLayer(nn.Module):
    """
    True Relation-Aware Layer.
    Uses a specific Weight Matrix (W_r) for each relation type.
    Equation: h_u = Tanh( W_0 * h_u + Sum_r( Sum_v( W_r * h_v ) ) )
    This is 'fat' because it has params: num_relations * dim * dim.
    """
    def __init__(self, in_dim, out_dim, num_rels):
        super().__init__()
        self.in_dim = in_dim
        self.out_dim = out_dim
        self.num_rels = num_rels
        
        # The 'Fat' Part: A unique weight matrix for every relation type
        # We use nn.ParameterTensor for efficient lookup
        self.weight = nn.Parameter(torch.Tensor(num_rels, in_dim, out_dim))
        
        # Self-loop weight (how much the node keeps its own identity)
        self.w_self = nn.Linear(in_dim, out_dim)
        
        # Initialization
        nn.init.xavier_uniform_(self.weight)

    def forward(self, x, adj_dict):
        # x: [num_nodes, in_dim]
        
        # 1. Start with Self-Loop info
        out = self.w_self(x)
        
        # 2. Iterate over relations (Relation-Aware Aggregation)
        #  
        for rel_type, adj in adj_dict.items():
            # Safety check for relation index
            if rel_type >= self.num_rels: continue 
            
            # A. Retrieve the specific Weight Matrix for this relation
            w_r = self.weight[rel_type] # [in_dim, out_dim]
            
            # B. Transform node features specifically for this relation
            #    (Dense Matrix Mul -> GPU Efficient)
            x_transformed = torch.matmul(x, w_r) # [num_nodes, out_dim]
            
            # C. Propagate using the sparse adjacency
            #    (Sparse Matrix Mul)
            #    Aggregates the transformed neighbors
            out_r = torch.sparse.mm(adj, x_transformed)
            
            # Add to accumulator
            out = out + out_r
            
        return torch.tanh(out)

class TrueKGCN(nn.Module):
    def __init__(self, num_nodes, num_relations, embedding_dim=64, n_layers=2):
        super().__init__()
        self.node_emb = nn.Embedding(num_nodes, embedding_dim)
        
        # Stack of RGCN Layers
        self.layers = nn.ModuleList([
            RGCNLayer(embedding_dim, embedding_dim, num_relations)
            for _ in range(n_layers)
        ])
        
        nn.init.xavier_uniform_(self.node_emb.weight)

    def forward(self, adj_dict):
        x = self.node_emb.weight
        for layer in self.layers:
            x = layer(x, adj_dict)
        return x

def build_relational_adjacency(kg_edges_df, num_nodes):
    # Same logic, just builds the sparse matrices
    adj_dict = {}
    rel_types = kg_edges_df['RelType'].unique()
    
    for r in rel_types:
        sub_df = kg_edges_df[kg_edges_df['RelType'] == r]
        src = sub_df['Src'].values
        dst = sub_df['Dst'].values
        indices = np.vstack((src, dst))
        values = np.ones(len(src))
        
        i = torch.LongTensor(indices)
        v = torch.FloatTensor(values)
        shape = (num_nodes, num_nodes)
        
        adj = torch.sparse_coo_tensor(i, v, shape).coalesce()
        
        # Row-Normalize
        deg = torch.sparse.sum(adj, dim=1).to_dense()
        deg_inv = deg.pow(-1)
        deg_inv[deg_inv == float('inf')] = 0
        
        adj = torch.sparse_coo_tensor(i, deg_inv[i[0]], shape).to(device)
        adj_dict[r] = adj
        
    return adj_dict

# ----------------------------
# 4. Feedback Training Loop (Unchanged Workflow)
# ----------------------------
LR, EPOCHS, BATCH_SIZE = 0.001, 50, 2048
MAX_RETRIES = 10 

MAIN_TARGET_HIGH = (0.45, 0.55)
MAIN_TARGET_LOW  = (0.15, 0.25)
NICHE_TARGET_HIGH = (0.35, 0.45)
NICHE_TARGET_LOW  = (0.35, 0.45)

w_main_vals = [1.5, 1.0, 0.1]
w_niche_vals = [2.0, 1.0, 0.3]

prev_model_state = None
prev_node_map = {} 

print(f"{'Slice Pair':<13} | {'Users':<6} | {'Recall':<8} | {'NDCG':<8} | {'EO':<8} | {'DP':<8} | {'Gini':<8}")
print("-" * 105)

slice_nums = sorted(ratings['SliceNum'].unique())

for s in slice_nums[:-1]: 
    train_df = ratings[ratings['SliceNum'] == s].copy()
    test_df  = ratings[ratings['SliceNum'] == s+1].copy()
    
    user_map_df = pd.read_csv(f"slice_metadata/kg/user_map_slice_{s}.csv")
    movie_map_df = pd.read_csv(f"slice_metadata/kg/movie_map_slice_{s}.csv")
    kg_edges_df = pd.read_csv(f"slice_metadata/kg/kg_edges_slice_{s}.csv")
    
    u_map = dict(zip(user_map_df.UserID, user_map_df.NodeID))
    i_map = dict(zip(movie_map_df.MovieID, movie_map_df.NodeID))
    
    current_node_lookup = {}
    for k, v in u_map.items(): current_node_lookup[f"u_{k}"] = v
    for k, v in i_map.items(): current_node_lookup[f"m_{k}"] = v

    user_labels = pd.read_csv(f"slice_metadata/user_labels_slice_{s}.csv").set_index('UserID')['UserType'].to_dict()
    movie_tiers = pd.read_csv(f"slice_metadata/movie_pop_tiers_slice_{s}.csv").set_index('MovieID')['PopTier'].to_dict()

    max_node_id = max(kg_edges_df['Src'].max(), kg_edges_df['Dst'].max()) + 1
    
    # CRITICAL: Determine max relation ID for weight matrix dimensioning
    max_rel_id = kg_edges_df['RelType'].max() + 1
    
    adj_dict = build_relational_adjacency(kg_edges_df, max_node_id)
    
    train_df = train_df[train_df['UserID'].isin(u_map) & train_df['MovieID'].isin(i_map)]
    train_user_tensor = torch.LongTensor(train_df['UserID'].map(u_map).values).to(device)
    train_item_tensor = torch.LongTensor(train_df['MovieID'].map(i_map).values).to(device)
    num_train_samples = len(train_user_tensor)

    tier_idx_map = {'low': 0, 'medium': 1, 'high': 2}
    u_type_idx_map = {'mainstream': 0, 'niche': 1}
    
    item_tier_tensor = torch.zeros(max_node_id, dtype=torch.long, device=device)
    for mid, node_id in i_map.items():
        tier = movie_tiers.get(mid, 'low')
        item_tier_tensor[node_id] = tier_idx_map[tier]

    user_type_tensor = torch.zeros(max_node_id, dtype=torch.long, device=device)
    for uid, node_id in u_map.items():
        ut = user_labels.get(uid, 'niche')
        user_type_tensor[node_id] = u_type_idx_map[ut]

    retry_count = 1
    success = False
    
    while retry_count <= MAX_RETRIES and not success:
        # INSTANTIATE TRUE KGCN
        model = TrueKGCN(max_node_id, max_rel_id).to(device)
        
        if prev_model_state:
            with torch.no_grad():
                # Load Layer Weights (The 'Fat' parts)
                # We iterate layers to load W_r safely
                for i, layer in enumerate(model.layers):
                    w_name = f'layers.{i}.weight'
                    ws_name = f'layers.{i}.w_self.weight'
                    bs_name = f'layers.{i}.w_self.bias'
                    
                    if w_name in prev_model_state:
                        # Slice to fit in case num_rels changed (unlikely but safe)
                        prev_w = prev_model_state[w_name]
                        curr_rels = layer.weight.shape[0]
                        prev_rels = prev_w.shape[0]
                        sz = min(curr_rels, prev_rels)
                        layer.weight.data[:sz] = prev_w[:sz].to(device)
                        
                    if ws_name in prev_model_state:
                        layer.w_self.weight.data = prev_model_state[ws_name].to(device)
                        layer.w_self.bias.data = prev_model_state[bs_name].to(device)

                # Load node embeddings
                prev_nodes = prev_model_state['node_emb.weight'].to(device)
                for entity_key, curr_id in current_node_lookup.items():
                    if entity_key in prev_node_map:
                        prev_id = prev_node_map[entity_key]
                        if prev_id < prev_nodes.shape[0]:
                            model.node_emb.weight.data[curr_id] = prev_nodes[prev_id]

        optimizer = optim.Adam(model.parameters(), lr=LR)
        weight_matrix = torch.tensor([w_main_vals, w_niche_vals], dtype=torch.float, device=device)
        
        model.train()
        
        # Optimization: Move negative sampling pool to GPU
        all_item_nodes = torch.LongTensor(list(i_map.values())).to(device)
        
        for epoch in range(EPOCHS):
            rand_indices = torch.randint(0, len(all_item_nodes), (num_train_samples,), device=device)
            neg_nodes = all_item_nodes[rand_indices]
            
            idxs = torch.randperm(num_train_samples, device=device)
            
            for i in range(0, num_train_samples, BATCH_SIZE):
                b = idxs[i:i+BATCH_SIZE]
                
                u_b = train_user_tensor[b]
                p_b = train_item_tensor[b]
                n_b = neg_nodes[b]
                
                final_emb = model(adj_dict)
                
                pos_scores = (final_emb[u_b] * final_emb[p_b]).sum(1)
                neg_scores = (final_emb[u_b] * final_emb[n_b]).sum(1)
                bpr = -torch.nn.functional.logsigmoid(pos_scores - neg_scores)
                
                w_p = weight_matrix[user_type_tensor[u_b], item_tier_tensor[p_b]]
                loss = (bpr * w_p).mean()
                
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
        
        metrics, count = compute_metrics(
            model, adj_dict, test_df, train_df, 
            u_map, i_map, user_labels, movie_tiers
        )
        
        if metrics:
            m_h = metrics['RawExposure']['mainstream']['high']
            m_l = metrics['RawExposure']['mainstream']['low']
            n_h = metrics['RawExposure']['niche']['high']
            n_l = metrics['RawExposure']['niche']['low']
            
            needs_retry = False
            if m_h < MAIN_TARGET_HIGH[0]: w_main_vals[2] += 0.05; needs_retry = True
            elif m_h > MAIN_TARGET_HIGH[1]: w_main_vals[2] = max(0, w_main_vals[2] - 0.05); needs_retry = True
            if m_l < MAIN_TARGET_LOW[0]: w_main_vals[0] += 0.2; needs_retry = True
            elif m_l > MAIN_TARGET_LOW[1]: w_main_vals[0] = max(0, w_main_vals[0] - 0.2); needs_retry = True
            if n_h < NICHE_TARGET_HIGH[0]: w_niche_vals[2] += 0.05; needs_retry = True
            elif n_h > NICHE_TARGET_HIGH[1]: w_niche_vals[2] = max(0, w_niche_vals[2] - 0.05); needs_retry = True
            if n_l < NICHE_TARGET_LOW[0]: w_niche_vals[0] += 0.2; needs_retry = True
            elif n_l > NICHE_TARGET_LOW[1]: w_niche_vals[0] = max(0, w_niche_vals[0] - 0.2); needs_retry = True

            if not needs_retry or retry_count == MAX_RETRIES:
                print(f"{s} -> {s+1:<8} | {count:<6} | {metrics['Recall']:.4f}    | {metrics['NDCG']:.4f}    | {metrics['EO']:.4f}    | {metrics['DP']:.4f}    | {metrics['Gini']:.4f}")
                w_m_str = "/".join([f"{v:.1f}" for v in w_main_vals])
                w_n_str = "/".join([f"{v:.1f}" for v in w_niche_vals])
                print(f"PopExposure: {metrics['PopExposure']} W: {{M:{w_m_str}, N:{w_n_str}}}")
                
                success = True
                prev_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                prev_node_map = current_node_lookup.copy()
            else:
                retry_count += 1
        else:
            break

Using device: cuda
Slice Pair    | Users  | Recall   | NDCG     | EO       | DP       | Gini    
---------------------------------------------------------------------------------------------------------
1 -> 2        | 96     | 0.1612    | 0.1093    | 0.0589    | 0.1218    | 0.6049
PopExposure: {'mainstream': {'low': '20.55%', 'medium': '35.47%', 'high': '43.98%'}, 'niche': {'low': '38.82%', 'medium': '30.29%', 'high': '30.90%'}} W: {M:1.9/1.0/0.3, N:2.2/1.0/0.8}
2 -> 3        | 117    | 0.1629    | 0.1032    | 0.0501    | 0.1373    | 0.5719
PopExposure: {'mainstream': {'low': '15.37%', 'medium': '36.98%', 'high': '47.65%'}, 'niche': {'low': '35.97%', 'medium': '26.99%', 'high': '37.04%'}} W: {M:2.1/1.0/0.3, N:2.2/1.0/0.8}
3 -> 4        | 166    | 0.1823    | 0.1008    | 0.0492    | 0.1244    | 0.5867
PopExposure: {'mainstream': {'low': '17.71%', 'medium': '31.32%', 'high': '50.97%'}, 'niche': {'low': '36.38%', 'medium': '23.96%', 'high': '39.66%'}} W: {M:2.1/1.0/0.3, N:2.2/1.0/0.9}
4 